# 🧪 Automated Psychometric Scale Development Pipeline

**Single-notebook master pipeline.** Each phase reads the previous phase's Excel output and writes a richer Excel file — creating a complete, auditable chain from construct definition to a human-review-ready item pool.

---

## Background

This pipeline was originally conceived and developed under the name **Project Majdal** by **Tariq Shaban (Inubilum Ltd)** as a practical implementation of emerging methods in AI-assisted psychometric scale development. The architecture draws directly on the pseudo-factor analysis (PFA) framework introduced by Guenole et al. (2024, 2025), the methodological evaluation of PFA by Suárez-Álvarez et al. (2026), the forced-choice measurement survey by Lee et al. (2025a), and the AI-powered item generation framework by Lee et al. (2025b).

> Guenole, N., Samo, A., Campion, J. K., Meade, A., Sun, T., & Oswald, F. (2024, February). *Pseudo-discrimination parameters from language embeddings*. Presented February 2024.

> Guenole, N., D’Urso, E. D., Samo, A., Sun, T., & Haslbeck, J. M. (2025, March). Enhancing scale development: Pseudo factor analysis of language embedding similarity matrices.

> Suárez-Álvarez, J., Fernández-Alonso, R., & Muñiz, J. (2026). Evaluating pseudo-factor analysis as a tool for item selection in psychological test development. *Psicothema*, *38*(1), 1–12.

> Milano, N., Luongo, M., Ponticorvo, M., & Marocco, D. (2025). Semantic analysis of test items through large language model embeddings predicts a-priori factorial structure of personality tests. *Current Research in Behavioral Sciences*, *8*, 100168. https://doi.org/10.1016/j.crbeha.2025.100168

> Lee, P., Son, M., & Jia, Z. (2025). AI-powered automatic item generation for psychological tests: A conceptual framework for an LLM-based multi-agent AIG system. *Journal of Business and Psychology*, 1–29.

> Lee, P., Son, M., Zhou, S., Joo, S., Jia, Z., & Cheng, V. (2025). The journey of forced choice measurement over 80 years: Past, present, and future. Organizational Research Methods, 28(4), 680-722.

> Lorenzo-Seva, U., & ten Berge, J. M. F. (2006). Tucker's congruence coefficient as a meaningful index of factor similarity. *Methodology: European Journal of Research Methods for the Behavioral and Social Sciences*, *2*(2), 57–64. https://doi.org/10.1027/1614-2241.2.2.57

---

## Pipeline Architecture

```
scales.json (input)
    │
    ├── PHASE 1 │ Behavioral Domain Generation and Validation → 01_behavioral_domains.xlsx
    ├── PHASE 2 │ Item Generation                             → 02_generated_items.xlsx
    ├── PHASE 3 │ Readability & Bias Analysis                 → 03_readability_bias.xlsx
    ├── PHASE 4 │ Content Validity (LLM SMEs)                 → 04_content_validity.xlsx
    └── PHASE 5 │ Pseudo-Factor Analysis and Final Review     → 05_pseudo_factor_analysis.xlsx
```

---

## Human Review Integration

Each phase that applies an automated pass/fail system also writes a **`human_review_pass`** column (default `True`) and a **`human_comments`** column. Reviewers open the Excel output after each phase and:

1. Change any `human_review_pass` value to `False` to override a system decision.
2. Add free-text reasoning in `human_comments`.

The **next phase always reads `human_review_pass`** as its processing filter — overriding the system `process_flag` where a human has intervened. This gives human experts full veto power at every stage without requiring any code changes.

---

## Before Running

1. Copy `.env_template` → `.env` and fill in `OPENROUTER_API_KEY`
2. Ensure `pipeline_settings.json` and `scales.json` are in the same folder as this notebook
3. Install requirements (Cell 1 does this automatically)
4. Run **Section 0 — Setup** before any other section
5. Each subsequent phase can be re-run independently provided its input Excel file exists

---

*Pipeline version: Revised (March 2026)*  
*Author: Tariq Shaban — Inubilum Ltd*


In [ ]:
import subprocess, sys

_packages = [
    "textstat", "openai", "pandas", "openpyxl",
    "scikit-learn", "sentence-transformers",
    "factor-analyzer", "python-dotenv", "tqdm"
]

process = subprocess.Popen(
    [sys.executable, "-m", "pip", "install"] + _packages,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
print(f"\nReturn code: {process.returncode}")

# ─── SECTION 0: Setup & Shared Utilities ───────────────────────────────────

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os, json, time, re, logging, warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import textstat
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from openai import OpenAI

warnings.filterwarnings("ignore")

# ── Working directory — always the folder containing this notebook ─────────────
def _get_notebook_dir() -> Path:
    # VS Code sets this variable automatically
    if "__vsc_ipynb_file__" in globals():
        return Path(__vsc_ipynb_file__).parent
    # Jupyter Lab / Notebook classic — try ipynbname
    try:
        import ipynbname
        return ipynbname.path().parent
    except Exception:
        pass
    # Final fallback — current working directory
    return Path.cwd()

NOTEBOOK_DIR = _get_notebook_dir()
os.chdir(NOTEBOOK_DIR)
print(f"✓ Working directory: {Path.cwd()}")

# ── Load .env ─────────────────────────────────────────────────────────────────
load_dotenv()

# ── Load pipeline_settings.json ───────────────────────────────────────────────
SETTINGS_FILE = Path("pipeline_settings.json")
assert SETTINGS_FILE.exists(), (
    f"Cannot find pipeline_settings.json in {Path.cwd()}. "
    "Make sure it is in the same folder as the notebook."
)
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)

# ── Output directory ──────────────────────────────────────────────────────────
BASE_OUTPUT_DIR = Path(os.getenv("BASE_OUTPUT_DIR", CFG["project"]["base_output_dir"]))
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────────
log_level = getattr(logging, CFG["project"].get("log_level", "INFO"))
logging.basicConfig(
    level=log_level,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("majdal")

# ── OpenRouter client ─────────────────────────────────────────────────────────
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
assert OPENROUTER_API_KEY, (
    "OPENROUTER_API_KEY is not set. "
    "Add it to your .env file: OPENROUTER_API_KEY=sk-or-..."
)

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    default_headers={
        "HTTP-Referer": "https://inubilum.io",
        "X-Title": os.getenv("OPENROUTER_APP_NAME", "Project Majdal")
    }
)

# ── Shared LLM call with exponential-backoff retry ────────────────────────────
def llm_call(
    prompt: str,
    model: str,
    max_tokens: int = 1000,
    temperature: float = 0.7,
    retries: int = 3,
    system: str = "You are a psychometric expert assistant."
) -> str:
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = (2 ** attempt) + np.random.uniform(0, 1)
            logger.warning(
                f"LLM call failed (attempt {attempt+1}/{retries}): {e}. "
                f"Retrying in {wait:.1f}s"
            )
            time.sleep(wait)
    raise RuntimeError(f"LLM call failed after {retries} attempts")

# ── Text cleaning helper ──────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not text:
        return text
    text = text.replace("**", "").replace("*", "")
    text = text.strip("[]")           # remove wrapping square brackets
    text = re.sub(r'^\[|\]$', '', text.strip())  # catch any remaining
    text = " ".join(text.split())
    return text.strip(".,;:!?- ")

# ── Scale normalisation ───────────────────────────────────────────────────────
SCALE_DEFAULTS = CFG["scale_defaults"]

def normalise_scale(s: Dict) -> Dict:
    """Map any scale JSON format to the standard field names used across all phases."""
    return {
        "scale_name":               s.get("scale_name")           or s.get("name", ""),
        "construct_definition":     s.get("construct_definition") or s.get("definition", ""),
        "high_anchor":              s.get("high_anchor")          or s.get("high_behaviors", ""),
        "low_anchor":               s.get("low_anchor")           or s.get("low_behaviors", ""),
        "measure_type":             s.get("measure_type")         or s.get("measure", "Trait"),
        "discriminant_description": s.get("discriminant_description", ""),
        "target_population":        s.get("target_population",    SCALE_DEFAULTS["target_population"]),
        "target_environment":       s.get("target_environment",   SCALE_DEFAULTS["target_environment"]),
        "reading_level_target":     s.get("reading_level_target", SCALE_DEFAULTS["reading_level_target"]),
        "response_format":          s.get("response_format",      SCALE_DEFAULTS["response_format"]),
        **{k: v for k, v in s.items() if k not in {
            "scale_name", "name", "construct_definition", "definition",
            "high_anchor", "high_behaviors", "low_anchor", "low_behaviors",
            "measure_type", "measure", "discriminant_description",
            "target_population", "target_environment",
            "reading_level_target", "response_format"
        }}
    }

# ── Load and normalise scales ─────────────────────────────────────────────────
scales_json_path     = Path(os.getenv("SCALES_JSON",     CFG["project"]["scales_json"]))
all_scales_json_path = Path(os.getenv("ALL_SCALES_JSON", CFG["project"]["all_scales_reference_json"]))

assert scales_json_path.exists(),     f"scales JSON not found: {scales_json_path}"
assert all_scales_json_path.exists(), f"all_scales JSON not found: {all_scales_json_path}"

with open(scales_json_path, "r", encoding="utf-8") as f:
    SCALES_DATA = json.load(f)
with open(all_scales_json_path, "r", encoding="utf-8") as f:
    ALL_SCALES_DATA = json.load(f)

TARGET_SCALES: List[Dict]  = [normalise_scale(s) for s in SCALES_DATA["scales"]]
ALL_SCALES_REF: List[Dict] = ALL_SCALES_DATA["scales"]

ALL_SCALES_LOOKUP: Dict[str, str] = {}
for s in ALL_SCALES_REF:
    name = s.get("name") or s.get("scale_name", "")
    defn = s.get("definition") or s.get("construct_definition", "")
    if name:
        ALL_SCALES_LOOKUP[name] = defn

ALL_SCALES_LIST_STR = "\n".join(
    f"- {name}: {defn[:120]}..." if len(defn) > 120 else f"- {name}: {defn}"
    for name, defn in ALL_SCALES_LOOKUP.items()
)

scale_params_lookup: Dict[str, Dict] = {s["scale_name"]: s for s in TARGET_SCALES}

# ── Summary ───────────────────────────────────────────────────────────────────
logger.info(f"✓ Loaded {len(TARGET_SCALES)} target scales from {scales_json_path.name}")
logger.info(f"✓ Loaded {len(ALL_SCALES_LOOKUP)} reference scales from {all_scales_json_path.name}")
logger.info(f"✓ Output directory: {BASE_OUTPUT_DIR.resolve()}")

print("\n🟢 Setup complete. Ready to run pipeline phases.")
print(f"\nTarget scales ({len(TARGET_SCALES)}):")
for s in TARGET_SCALES:
    print(f"  • {s['scale_name']}  |  env: {s['target_environment']}  |  "
          f"pop: {s['target_population'].get('age_range','?')} / "
          f"{s['target_population'].get('education_level','?')}")

# ─── PHASE 1: Behavioral Domain Generation and Validation ──────────────────

**Input:** `scales.json`  
**Output:** `01_behavioral_domains.xlsx`  
**Sheets:** `Behavioral_Domains`, `Validation_Summary`, `Scale_Parameters`, `Run_Log`

---

### What this phase does

Phase 1 reads each construct definition from `scales.json` and performs two operations in sequence:

1. **Generation** — the LLM generates observable behavioral examples — concrete descriptions of what someone high or low on the trait actually *does* in context. These examples form the generative substrate from which all downstream items are produced. The `occurrence_likelihood` field (low / moderate / high) reflects how frequently the behavior typically occurs in the target environment and population — **not** item difficulty in the psychometric sense.

2. **Validation** — a second LLM pass evaluates each generated example for three validity criteria:
   - **Construct validity** — does the example genuinely reflect the target construct?
   - **Face validity** — would it appear relevant to target respondents?
   - **Discriminant validity** — is it distinct from the excluded constructs listed in `discriminant_description`?

Examples that fail any criterion have `process_flag` set to `False`. Only examples with a valid flag proceed to item writing (Phase 2).

### System pass/fail rules (from `pipeline_settings.json`)

| Setting | Role |
|---|---|
| `examples_per_construct` | Target number of examples per construct |
| `min_valid_examples_threshold` | Minimum valid examples required per scale before raising a warning |
| `validation_criteria` | The three criteria evaluated per example |
| `retry_attempts` | Retries on API failure |

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `process_flag` | System-set | LLM validation result |
| `human_review_pass` | `True` | Override — set `False` to exclude an example the system passed, or correct a false failure |
| `human_comments` | *(blank)* | Record reasoning for any override |

**Phase 2 reads `human_review_pass`.** Review `01_behavioral_domains.xlsx → Behavioral_Domains` before running Phase 2. Pay particular attention to `validation_issues` for context on why the system flagged an example.


In [ ]:
P1         = CFG["phase_01_behavioral_domains"]
P1_GEN_MODEL  = P1["generation_model"]
P1_VAL_MODEL  = P1["validation_model"]
P1_OUT     = BASE_OUTPUT_DIR / P1["output_file"]

# ── Phase 1A: Behavioral Domain Generation ────────────────────────────────────

def generate_behavioral_examples(scale: Dict) -> List[Dict]:
    """Generate stratified behavioral domain examples for one scale."""

    prompt = f"""You are a psychometric expert generating behavioral examples for a personality assessment.

Construct: {scale['scale_name']}
Definition: {scale['construct_definition']}
High pole: {scale.get('high_anchor', 'N/A')}
Low pole: {scale.get('low_anchor', 'N/A')}
What this construct is NOT: {scale.get('discriminant_description', 'N/A')}
Target population: {json.dumps(scale.get('target_population', {}))}
Target environment: {scale.get('target_environment', 'General')}

Generate exactly {P1['examples_per_construct']} distinct, observable behavioral examples for this construct.
Each example must describe a recurring pattern of behavior — not a one-off event.

For each example assign an occurrence_likelihood level (high / moderate / low) reflecting
how commonly this behavior occurs in the target environment. Distribute across levels.

Format each example EXACTLY like this (one per block, blocks separated by ---):
EXAMPLE_ID: 1
BEHAVIOR: [concrete observable behavior, 1-2 sentences]
LEVEL: [high / moderate / low]
RATIONALE: [why this is a valid example of {scale['scale_name']}]
ENVIRONMENT_FIT: [why this behavior occurs in {scale.get('target_environment', 'the target environment')}]
POPULATION_FIT: [why this is relevant to the target population]
---"""

    raw = llm_call(
        prompt, P1_GEN_MODEL,
        max_tokens  = P1["generation_max_tokens"],
        temperature = P1["generation_temperature"],
        retries     = P1["retry_attempts"]
    )

    examples = []
    blocks   = [b.strip() for b in raw.split("---") if b.strip()]

    for i, section in enumerate(blocks):
        data = {
            "scale_name":       scale["scale_name"],
            "construct":        scale.get("construct_definition", ""),
            "target_environment": scale.get("target_environment", ""),
            "target_population":  json.dumps(scale["target_population"]),
            "process_flag":       True,
            "validation_issues":  ""
        }

        for line in section.split("\n"):
            line = line.strip()
            for key, field in [
                ("EXAMPLE_ID:",      "example_id"),
                ("BEHAVIOR:",        "behavior_description"),
                ("LEVEL:",           "occurrence_likelihood"),
                ("RATIONALE:",       "rationale"),
                ("ENVIRONMENT_FIT:", "environment_relevance"),
                ("POPULATION_FIT:",  "population_relevance"),
            ]:
                if line.upper().startswith(key):
                    data[field] = clean_text(line[len(key):].strip())

        # Build ID from scale's id field + sequential domain number
        scale_id = scale.get("id", scale["scale_name"][:3].upper())
        if "example_id" not in data:
            data["example_id"] = f"{scale_id}_D{i+1:02d}"
        else:
            # Reformat whatever the LLM returned to match our convention
            data["example_id"] = f"{scale_id}_D{len(examples)+1:02d}"

        if "behavior_description" in data and data["behavior_description"]:
            examples.append(data)

    return examples


# ── Phase 1B: Domain Validation ───────────────────────────────────────────────

def validate_examples_for_scale(scale_name: str, examples: List[Dict], scale: Dict) -> Dict[str, Dict]:
    """Batch-validate up to 10 examples at once to reduce API calls."""

    # Build a numbered list so we can match by position if ID matching fails
    numbered = []
    for idx, e in enumerate(examples):
        numbered.append({
            "n":          idx + 1,
            "example_id": e.get("example_id", f"item_{idx+1}"),
            "behavior":   e.get("behavior_description", ""),
            "level":      e.get("occurrence_likelihood", "")
        })

    batch_str = json.dumps(numbered, indent=2)

    prompt = f"""Review these behavioral examples for the construct: {scale_name}

Construct Definition: {scale.get('construct_definition', '')}
What it is NOT: {scale.get('discriminant_description', 'N/A')}

For each example evaluate:
1. construct_validity: Does it truly measure the target construct?
2. face_validity: Does it appear relevant to target users?
3. discriminant_validity: Is it distinct from excluded constructs?

Examples to evaluate:
{batch_str}

You MUST return exactly {len(numbered)} JSON objects, one per example, separated by ---.
Use the EXACT example_id values provided above — do not shorten or modify them.

{{"example_id": "<exact id from above>", "valid": true/false, "issues": "<brief reason or empty string>"}}
---"""

    raw = llm_call(
        prompt, P1_VAL_MODEL,
        max_tokens  = P1["validation_max_tokens"],
        temperature = P1["validation_temperature"],
        retries     = P1["retry_attempts"]
    )

    # Parse results — try ID match first, fall back to positional match
    results = {}
    parsed_blocks = []
    for block in raw.split("---"):
        block = block.strip()
        if not block:
            continue
        try:
            cleaned = re.sub(r'```json|```', '', block).strip()
            obj = json.loads(cleaned)
            parsed_blocks.append(obj)
            # Primary match: exact example_id
            eid = str(obj.get("example_id", ""))
            if eid:
                results[eid] = {
                    "valid":  bool(obj.get("valid", True)),
                    "issues": obj.get("issues", "")
                }
        except Exception:
            pass

    # Fallback: positional match for any examples still unmatched
    for idx, e in enumerate(examples):
        eid = str(e.get("example_id", ""))
        if eid not in results and idx < len(parsed_blocks):
            obj = parsed_blocks[idx]
            results[eid] = {
                "valid":  bool(obj.get("valid", True)),
                "issues": obj.get("issues", "") + " [matched positionally]"
            }

    return results


# ── Run Phase 1 ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"PHASE 1: Behavioral Domain Generation and Validation")
print(f"Generation model : {P1_GEN_MODEL}")
print(f"Validation model : {P1_VAL_MODEL}")
print(f"Scales           : {len(TARGET_SCALES)}")
print(f"{'='*60}\n")

# ── Step 1A: Generate behavioral examples ─────────────────────────────────────
all_examples: List[Dict] = []
run_log_p1:   List[Dict] = []

for scale in tqdm(TARGET_SCALES, desc="Generating examples"):
    scale_name = scale["scale_name"]
    t0 = time.time()
    try:
        examples = generate_behavioral_examples(scale)
        all_examples.extend(examples)
        status = "success"
        n      = len(examples)
    except Exception as e:
        logger.error(f"Phase 1 generation failed for '{scale_name}': {e}")
        examples = []
        status   = f"error: {e}"
        n        = 0

    run_log_p1.append({
        "scale_name":         scale_name,
        "phase":              "generation",
        "examples_generated": n,
        "status":             status,
        "duration_s":         round(time.time() - t0, 2),
        "timestamp":          datetime.now().isoformat()
    })
    print(f"  {'✓' if n > 0 else '✗'} {scale_name}: {n} examples")

print(f"\n  Total examples generated: {len(all_examples)}")

# ── Step 1B: Validate behavioral examples ─────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Validating examples...")
print(f"{'─'*60}")

df_examples = pd.DataFrame(all_examples)
scale_params_lookup = {s["scale_name"]: s for s in TARGET_SCALES}

validation_results_all: Dict[str, Dict] = {}
for scale_name, group in tqdm(df_examples.groupby("scale_name"), desc="Validating"):
    scale = scale_params_lookup.get(scale_name, {})
    examples = group.to_dict(orient="records")
    t0 = time.time()
    try:
        batch_size = 10
        for i in range(0, len(examples), batch_size):
            batch = examples[i:i+batch_size]
            batch_results = validate_examples_for_scale(scale_name, batch, scale)
            validation_results_all.update(batch_results)
        status = "success"
    except Exception as e:
        logger.error(f"Phase 1 validation failed for {scale_name}: {e}")
        status = f"error: {e}"

    run_log_p1.append({
        "scale_name": scale_name,
        "phase":      "validation",
        "examples":   len(examples),
        "status":     status,
        "duration_s": round(time.time()-t0, 2),
        "timestamp":  datetime.now().isoformat()
    })

# Apply validation results back to dataframe
def apply_validation(row):
    eid = str(row.get("example_id", ""))
    result = validation_results_all.get(eid, {"valid": True, "issues": "not reviewed"})
    row["process_flag"]      = result["valid"]
    row["validation_issues"] = result.get("issues", "")
    return row

df_examples = df_examples.apply(apply_validation, axis=1)

# Diagnostic — show any still not reviewed
not_reviewed = df_examples[df_examples["validation_issues"] == "not reviewed"]
if len(not_reviewed) > 0:
    print(f"⚠️  {len(not_reviewed)} examples still not reviewed:")
    print(not_reviewed[["scale_name", "example_id"]].to_string())
else:
    print("✓ All examples reviewed")

# Summary stats
summary_rows = []
for scale_name, group in df_examples.groupby("scale_name"):
    total = len(group)
    valid = group["process_flag"].sum()
    summary_rows.append({
        "scale_name":      scale_name,
        "total_examples":  total,
        "valid_examples":  int(valid),
        "invalid_examples": total - int(valid),
        "pass_rate_pct":   round(100*valid/total, 1) if total else 0
    })

df_summary = pd.DataFrame(summary_rows)
df_params  = pd.DataFrame(TARGET_SCALES)
df_log     = pd.DataFrame(run_log_p1)

# ── Human review columns ──────────────────────────────────────────────────────
df_examples["human_review_pass"] = True   # set False to exclude an example
df_examples["human_comments"]    = ""     # free-text rationale for any override

# ── Write Excel ───────────────────────────────────────────────────────────────
with pd.ExcelWriter(P1_OUT, engine="openpyxl") as writer:
    df_examples.to_excel(writer, sheet_name="Behavioral_Domains", index=False)
    df_summary .to_excel(writer, sheet_name="Validation_Summary", index=False)
    df_params  .to_excel(writer, sheet_name="Scale_Parameters",   index=False)
    df_log     .to_excel(writer, sheet_name="Run_Log",            index=False)

total_valid = df_examples["process_flag"].sum()
print(f"\n✅ Phase 1 complete.")
print(f"   {len(all_examples)} examples generated, {int(total_valid)} passed validation.")
print(f"📄 Output: {P1_OUT}")


# ─── PHASE 2: Item Generation ───────────────────────────────────────────────

**Input:** `01_behavioral_domains.xlsx` → sheet `Behavioral_Domains` (respects `human_review_pass`)  
**Output:** `02_generated_items.xlsx`  
**Sheets:** `Generated_Items`, `Generation_Summary`, `Run_Log`

---

### What this phase does

Phase 2 generates survey items — first-person "I" statements — from each validated behavioral example. Items are written at three trait levels (high / moderate / low) following the Goldberg item-writing principles: short, specific, present tense, plain vocabulary, and single-idea.

The `difficulty_target` column on each generated item reflects the trait level of the behavioral example it was derived from, not reading difficulty.

### System pass/fail rules (from `pipeline_settings.json`)

| Setting | Role |
|---|---|
| `items_per_difficulty` | `{"high": N, "moderate": N, "low": N}` — items generated per example per difficulty |
| `duplicate_removal_threshold` | Cosine similarity above which near-duplicate items are removed (0–1, default 0.85) |
| `retry_attempts` | Retries on API failure |

### Human review columns added to output

No automated pass/fail is applied in Phase 2 — all generated items are passed to Phase 3 for readability screening. However:

| Column | Default | Purpose |
|---|---|---|
| `human_review_pass` | `True` | Manually exclude an item before readability analysis |
| `human_comments` | *(blank)* | Notes on exclusion |

**Phase 3 reads `human_review_pass`.** This allows rejection of items on sight (e.g., obvious construct contamination or generation failure) without waiting for downstream analysis.


In [ ]:
# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

P3 = CFG["phase_02_item_generation"]
P3_MODEL = P3["model"]
P3_IN  = BASE_OUTPUT_DIR / CFG["phase_01_behavioral_domains"]["output_file"]
P3_OUT = BASE_OUTPUT_DIR / P3["output_file"]

assert P3_IN.exists(), f"Input not found: {P3_IN}. Run Phase 1 first."
df_p3_in = pd.read_excel(P3_IN, sheet_name="Behavioral_Domains")
# Respect human overrides from Phase 1
if "human_review_pass" in df_p3_in.columns:
    # Respect both validation process_flag and human overrides
    mask = (df_p3_in["human_review_pass"] != False)
    if "process_flag" in df_p3_in.columns:
        mask = mask & (df_p3_in["process_flag"] != False)
    df_p3_valid = df_p3_in[mask].copy()
    n_excl = len(df_p3_in) - len(df_p3_valid)
    print(f"✓ Phase 1 human review: {n_excl} examples excluded by human_review_pass")
else:
    df_p3_valid = df_p3_in.copy()
print(f"Using {len(df_p3_valid)} valid examples (of {len(df_p3_in)} total)")

items_per_diff = P3["items_per_difficulty"]

# Levels actually requested (count > 0) — used by prompt builder and parser
ACTIVE_LEVELS = [lvl for lvl in ("high", "moderate", "low") if items_per_diff.get(lvl, 0) > 0]

def generate_items_for_example(example: Dict, scale: Dict) -> List[Dict]:
    """Generate stratified I-statement items for one behavioral example."""

    difficulty_lines = []
    format_lines     = []
    for lvl in ACTIVE_LEVELS:
        label = lvl.upper()
        n     = items_per_diff[lvl]
        difficulty_lines.append(f"- {n} items for people {label} on this trait")
        format_lines.append(f"{label} DIFFICULTY\n* I [item — max 9 words]\n...")

    difficulty_request = "\n".join(difficulty_lines)
    format_block       = "\n".join(format_lines)

    example_id = str(example.get("example_id", "")).replace("*","").replace("/","").replace(":","")

    prompt = f"""You are writing personality survey items for the construct: {scale['scale_name']}

Construct Definition: {scale['construct_definition']}
Target Population: {json.dumps(scale.get('target_population', {}))}
Setting: {scale.get('target_environment', 'General')}
Behavioral domain: "{example.get('behavior_description', '')}"
Domain likelihood in target environment: {example.get('occurrence_likelihood', 'moderate')}

Create exactly:
{difficulty_request}

CRITICAL RULE — DOMAIN SPECIFICITY:
Every item you write MUST be rooted in the specific behavioral domain above.
If you removed the domain description and replaced it with a different domain,
the item should NO LONGER make sense or fit. Generic items that could apply to
ANY domain of {scale['scale_name']} are NOT acceptable.

For example, if the domain is "Using metaphors to explain ideas to colleagues":
GOOD: "I use analogies to make hard ideas clear"  ← specific to this domain
BAD:  "I see patterns others miss in problems"     ← could come from any domain

ITEM WRITING RULES — follow all of these strictly:
1. Write as "I" statements in the present tense (e.g., "I enjoy solving hard problems")
2. MAXIMUM 9 words per item — count every word including "I"
3. One idea only — never use "and", "but", or "or" to join two thoughts
4. No negations — never write "I don't", "I never", "I am not"
5. Plain everyday words only — if a 16-year-old might not know the word, replace it
6. No abstract nouns — avoid words like "efficiency", "integrity", "competence"
7. Describe observable behavior or clear personal tendency, not beliefs or values
8. Must be tightly anchored to the specific behavioral domain above — not the construct in general
9. Each item on its own line starting with "* "
10. No brackets [], quotes, or formatting in the statements
11. US spelling only
12. Each item must measure {scale['scale_name']} ONLY — if a neutral observer could \
plausibly attribute the behavior to a different construct, rewrite it
13. No two items may express the same idea with different words — each item must \
describe a meaningfully distinct behavior within this domain

GOOD examples (short, plain, domain-specific, each one distinct):
* I keep my desk tidy
* I finish tasks ahead of time
* I follow rules carefully
* I like to help others
* I speak up in meetings

BAD examples (generic, duplicate in meaning, or construct-contaminated — never write these):
* I consistently demonstrate a strong commitment to maintaining high standards
* I connect ideas from different fields together  [TOO GENERIC — applies to any abstract thinking domain]
* I see patterns others miss  [TOO GENERIC — not anchored to this specific domain]
* I think about problems in different ways  [TOO GENERIC]
* I help my team stay organized [CONTAMINATED — measures both Altruism and Conscientiousness]

Format EXACTLY (include ONLY the sections listed above — do not add sections not requested):
{format_block}"""

    raw = llm_call(
        prompt, P3_MODEL,
        max_tokens  = P3["max_tokens"],
        temperature = P3["temperature"],
        retries     = P3["retry_attempts"]
    )

    items = []
    current_diff = None
    item_counter = 1

    for line in raw.split("\n"):
        line = line.strip()
        lu   = line.upper()

        if lu == "HIGH DIFFICULTY":
            current_diff = "high"     if "high"     in ACTIVE_LEVELS else None
            continue
        if lu == "MODERATE DIFFICULTY":
            current_diff = "moderate" if "moderate" in ACTIVE_LEVELS else None
            continue
        if lu == "LOW DIFFICULTY":
            current_diff = "low"      if "low"      in ACTIVE_LEVELS else None
            continue

        if current_diff and line.startswith("* "):
            already = sum(1 for i in items if i["difficulty_target"] == current_diff)
            if already >= items_per_diff[current_diff]:
                continue
            text = clean_text(line[2:].strip())
            if text.startswith("I ") and len(text) > 8:
                items.append({
                    "item_id":               f"{example_id}_I{item_counter:02d}",
                    "scale_name":            example.get("scale_name", ""),
                    "item_text":             text,
                    "difficulty_target":     current_diff,
                    "source_example":        example.get("example_id", ""),
                    "example_behavior":      example.get("behavior_description", ""),
                    "occurrence_likelihood": example.get("occurrence_likelihood", ""),
                    "process_flag":          True
                })
                item_counter += 1

    return items


def flag_near_duplicates(items: List[Dict], threshold: float = 0.85) -> List[Dict]:
    """Flag near-duplicate items for human review instead of removing them."""
    if len(items) < 2:
        return items
    texts = [i["item_text"] for i in items]
    try:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        mat = vec.fit_transform(texts)
        sim = sk_cosine(mat)
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                if sim[i, j] > threshold:
                    # Flag both items — human decides which to keep
                    items[i]["duplicate_flag"]    = True
                    items[i]["duplicate_of"]      = items[j]["item_id"]
                    items[j]["duplicate_flag"]    = True
                    items[j]["duplicate_of"]      = items[i]["item_id"]
                    items[i]["duplicate_similarity"] = round(float(sim[i, j]), 3)
                    items[j]["duplicate_similarity"] = round(float(sim[i, j]), 3)
    except Exception as e:
        logger.warning(f"Duplicate flagging failed: {e}")
    return items


# ── Run Phase 3 ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"PHASE 2: Item Generation")
print(f"Model  : {P3_MODEL}")
print(f"Valid examples : {len(df_p3_valid)}")
print(f"Active difficulty levels: {ACTIVE_LEVELS}")
print(f"Items per level: { {lvl: items_per_diff[lvl] for lvl in ACTIVE_LEVELS} }")
print(f"{'='*60}\n")

all_items:    List[Dict] = []
run_log_p3:   List[Dict] = []
scale_totals: Dict[str, int] = {}

for _, row in tqdm(df_p3_valid.iterrows(), total=len(df_p3_valid), desc="Examples"):
    example    = row.to_dict()
    scale_name = example.get("scale_name", "")
    scale      = scale_params_lookup.get(scale_name, {})
    t0 = time.time()

    try:
        items  = generate_items_for_example(example, scale)
        all_items.extend(items)
        status = "success"
        n      = len(items)
    except Exception as e:
        logger.error(f"Phase 3 failed for {example.get('example_id')}: {e}")
        status = f"error: {e}"
        n      = 0

    scale_totals[scale_name] = scale_totals.get(scale_name, 0) + n
    print(
        f"  {'✓' if n > 0 else '✗'} {example.get('example_id')} "
        f"→ {n} items generated  "
        f"[{scale_name} running total: {scale_totals[scale_name]}]"
    )

    run_log_p3.append({
        "example_id":      example.get("example_id"),
        "scale_name":      scale_name,
        "items_generated": n,
        "status":          status,
        "duration_s":      round(time.time()-t0, 2),
        "timestamp":       datetime.now().isoformat()
    })
    time.sleep(P3["inter_example_delay_seconds"])

# ── Flag near-duplicates within each scale (no items removed) ─────────────────
print(f"\n{'='*60}")
print("Duplicate Flagging (no items removed — flagged for human review)")
print(f"{'='*60}")

flagged_items: List[Dict] = []
for scale_name in df_p3_valid["scale_name"].unique():
    scale_items = [i for i in all_items if i["scale_name"] == scale_name]
    flagged     = flag_near_duplicates(scale_items, P3["duplicate_removal_threshold"])
    flagged_items.extend(flagged)
    n_flagged = sum(1 for i in flagged if i.get("duplicate_flag"))
    print(f"  {scale_name}: {len(flagged)} items, {n_flagged} flagged as near-duplicates")

# ── Build dataframe — fill missing duplicate columns with defaults ─────────────
df_items = pd.DataFrame(flagged_items)
if not df_items.empty:
    df_items["duplicate_flag"]       = df_items.get("duplicate_flag",       pd.Series(False, index=df_items.index)).fillna(False)
    df_items["duplicate_of"]         = df_items.get("duplicate_of",         pd.Series("",    index=df_items.index)).fillna("")
    df_items["duplicate_similarity"] = df_items.get("duplicate_similarity", pd.Series("",    index=df_items.index)).fillna("")

    gen_summary = df_items.groupby(
        ["scale_name", "difficulty_target"]
    ).size().reset_index(name="item_count")
    print(f"\nFinal item counts by scale and difficulty:")
    print(gen_summary.to_string(index=False))
else:
    gen_summary = pd.DataFrame()

# ── Human review columns ──────────────────────────────────────────────────────
# human_review_pass defaults True — set False to exclude an item before Phase 4
# Duplicate-flagged items should be prioritised for review
df_items["human_review_pass"] = True
df_items["human_comments"]    = ""

with pd.ExcelWriter(P3_OUT, engine="openpyxl") as writer:
    df_items        .to_excel(writer, sheet_name="Generated_Items",    index=False)
    gen_summary     .to_excel(writer, sheet_name="Generation_Summary", index=False)
    pd.DataFrame(run_log_p3).to_excel(writer, sheet_name="Run_Log",   index=False)

print(f"\n✅ Phase 3 complete. {len(flagged_items)} items across "
      f"{df_p3_valid['scale_name'].nunique()} scales.")
print(f"   {sum(1 for i in flagged_items if i.get('duplicate_flag'))} items flagged as near-duplicates for human review.")
print(f"📄 Output: {P3_OUT}")

# ─── PHASE 3: Readability & Bias Analysis ───────────────────────────────────

**Input:** `02_generated_items.xlsx` → sheet `Generated_Items` (respects `human_review_pass`)  
**Output:** `03_readability_bias.xlsx`  
**Sheets:** `Items_With_Readability`, `Readability_Summary`, `Bias_Summary`, `Simplified_Items_Log`

---

### What this phase does

Phase 4 subjects every item to two quality filters:

**Readability screening:** Six readability metrics are computed (Flesch-Kincaid grade, Flesch Reading Ease, Gunning Fog, Coleman–Liau, SMOG, Automated Readability Index). Items whose Flesch-Kincaid grade exceeds the `reading_level_hard_cap` are automatically simplified by the LLM (up to `max_simplification_attempts` times). The original wording is preserved in `original_item_text` for audit purposes.

**Bias detection:** Regular-expression pattern matching flags items containing gender, age, cultural, or socioeconomic language. Flagged items are not automatically excluded — they are passed forward for human review.

### System pass/fail rules (from `pipeline_settings.json`)

| Setting | Role |
|---|---|
| `reading_level_target` | Target FK grade (soft threshold — items above this are flagged) |
| `reading_level_hard_cap` | Items above this grade are simplified automatically |
| `max_simplification_attempts` | Maximum LLM simplification retries per item |
| `bias_categories` | Categories checked by pattern matching |

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `system_readability_pass` | System-set (`True`/`False`) — `False` if item remains above hard cap after all simplification attempts | Automated readability verdict |
| `human_review_pass` | `True` | Override — e.g., accept a borderline item or reject a bias-flagged item |
| `human_comments` | *(blank)* | Notes — especially important for bias-flagged items |

**Phase 4 reads `human_review_pass`.** Review `03_readability_bias.xlsx → Items_With_Readability` before running Phase 4 — particularly items in `Bias_Summary` and any with `above_reading_target = True`.


In [ ]:
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

P4     = CFG["phase_03_readability_bias"]
P4_IN  = BASE_OUTPUT_DIR / CFG["phase_02_item_generation"]["output_file"]
P4_OUT = BASE_OUTPUT_DIR / P4["output_file"]

assert P4_IN.exists(), f"Input not found: {P4_IN}. Run Phase 2 first."
df_p4_raw = pd.read_excel(P4_IN, sheet_name="Generated_Items")
# Respect human overrides from Phase 3
if "human_review_pass" in df_p4_raw.columns:
    df_p4 = df_p4_raw[df_p4_raw["human_review_pass"] == True].copy()
    n_excl = len(df_p4_raw) - len(df_p4)
    print(f"✓ Phase 2 human review: {n_excl} items excluded by human_review_pass")
else:
    df_p4 = df_p4_raw.copy()

BIAS_PATTERNS = {
    "gender":        [r'\b(he|him|his|she|her|hers|man|woman|male|female|guy|girl)\b'],
    "age":           [r'\b(young|old|elderly|senior|youth|teenager|millennial|boomer)\b'],
    "cultural":      [r'\b(foreign|native|immigrant|american|christian|muslim|jewish|western|eastern)\b'],
    "socioeconomic": [r'\b(poor|rich|wealthy|expensive|cheap|luxury|ghetto|privileged)\b']
}

def analyze_readability(text: str) -> Dict[str, float]:
    if not text or not text.strip():
        return {m: 0.0 for m in P4["readability_metrics"]}
    return {
        "flesch_kincaid_grade":        round(textstat.flesch_kincaid_grade(text), 3),
        "flesch_reading_ease":         round(textstat.flesch_reading_ease(text), 3),
        "gunning_fog":                 round(textstat.gunning_fog(text), 3),
        "coleman_liau_index":          round(textstat.coleman_liau_index(text), 3),
        "smog_index":                  round(textstat.smog_index(text), 3),
        "automated_readability_index": round(textstat.automated_readability_index(text), 3)
    }

def detect_bias(text: str) -> List[str]:
    flags = []
    tl = text.lower()
    for bias_type, patterns in BIAS_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, tl):
                flags.append(bias_type)
                break
    return flags

def simplify_item(text: str, scale_name: str, target_grade: int) -> str:
    prompt = f"""Simplify the wording of this personality survey item.

Construct being measured: {scale_name}
Original item: {text}

Your task is ONLY to simplify the words — do NOT change the meaning, content,
or intent of the original item. Every core idea in the original must be preserved.
If the original says "I organise my work carefully", the simplified version must
still be about organising work carefully — not a related but different behaviour.

GOOD simplifications (same meaning, simpler words):
* "I maintain a well-organized and structured workspace" → "I keep my workspace tidy"
* "I consistently follow through on my commitments" → "I do what I say I will do"
* "I demonstrate persistence when faced with obstacles" → "I keep going when things get hard"
* "I engage thoughtfully before reaching conclusions" → "I think before I decide"
* "I take considerable care when completing assigned tasks" → "I take care with my work"

BAD simplifications (meaning has changed or drifted — never do this):
* "I maintain a well-organized and structured workspace" → "I plan my day carefully" [WRONG — drifted to planning, not tidiness]
* "I consistently follow through on my commitments" → "I am a reliable person" [WRONG — too vague, lost the behavioral specificity]
* "I demonstrate persistence when faced with obstacles" → "I work hard every day" [WRONG — hard work is not the same as persistence under difficulty]
* "I engage thoughtfully before reaching conclusions" → "I listen to others carefully" [WRONG — introduced a new idea not in the original]

Rules:
1. Keep the "I" statement format in present tense
2. MAXIMUM 9 words — count every word including "I"
3. Replace long or complex words with shorter everyday equivalents
4. No negations — never write "I don't", "I never", "I am not"
5. No abstract nouns, brackets, quotes, or formatting
6. US spelling only
7. Must still clearly measure {scale_name} and nothing else
8. Reading level at or below Grade {target_grade}
9. Do NOT introduce new ideas, behaviours, or interpretations not in the original

Respond with ONLY the simplified item — no explanation, no punctuation at the end.

Original item to simplify: {text}"""

    resp = llm_call(
        prompt,
        P4["simplification_model"],
        max_tokens  = P4["simplification_max_tokens"],
        temperature = P4["simplification_temperature"]
    )
    for line in resp.split("\n"):
        line = clean_text(line)
        if line.startswith("I ") and 8 < len(line) < 200:
            return line
    return clean_text(resp.split("\n")[0])[:200]


# ── Run Phase 4 ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"PHASE 3: Readability & Bias Analysis")
print(f"Items to process : {len(df_p4)}")
print(f"Reading target   : Grade {P4['reading_level_target']}")
print(f"Hard cap         : Grade {P4['reading_level_hard_cap']}")
print(f"Max simplification attempts: {P4.get('max_simplification_attempts', 3)}")
print(f"{'='*60}\n")

readability_cols = P4["readability_metrics"]
simplified_log:  List[Dict] = []

# Initialise all new columns
for col in readability_cols:
    df_p4[col] = 0.0

df_p4["bias_flags"]           = ""
df_p4["bias_flag_count"]      = 0
df_p4["was_simplified"]       = False
df_p4["simplification_attempts"] = 0
df_p4["original_item_text"]   = ""
df_p4["above_reading_target"] = False

max_attempts = P4.get("max_simplification_attempts", 3)

for idx, row in tqdm(df_p4.iterrows(), total=len(df_p4), desc="Items"):
    item_id    = row.get("item_id", f"item_{idx}")
    text       = str(row.get("item_text", ""))
    scale_name = str(row.get("scale_name", ""))
    scale      = scale_params_lookup.get(scale_name, {})
    target_grade = int(scale.get("reading_level_target", P4["reading_level_target"]))

    # ── Initial readability scores ────────────────────────────────────────────
    scores   = analyze_readability(text)
    fk_grade = scores["flesch_kincaid_grade"]
    for col, val in scores.items():
        df_p4.at[idx, col] = val

    # ── Simplification retry loop ─────────────────────────────────────────────
    attempt = 0
    while fk_grade > P4["reading_level_hard_cap"] and attempt < max_attempts:
        attempt += 1
        try:
            # Each retry simplifies the CURRENT item_text — not the original
            current_text = str(df_p4.at[idx, "item_text"])
            new_text     = simplify_item(current_text, scale_name, target_grade)
            new_scores   = analyze_readability(new_text)

            # Log every attempt for audit trail
            simplified_log.append({
                "item_id":               item_id,
                "scale_name":            scale_name,
                "source_example":        row.get("source_example", ""),
                "occurrence_likelihood": row.get("occurrence_likelihood", ""),
                "attempt":               attempt,
                "input_text":            current_text,
                "simplified_text":       new_text,
                "input_fk_grade":        fk_grade,
                "new_fk_grade":          new_scores["flesch_kincaid_grade"]
            })

            # Preserve the very first original only
            if not df_p4.at[idx, "original_item_text"]:
                df_p4.at[idx, "original_item_text"] = text

            # Always update item_text and scores to reflect latest version
            df_p4.at[idx, "item_text"]               = new_text
            df_p4.at[idx, "was_simplified"]           = True
            df_p4.at[idx, "simplification_attempts"]  = attempt
            for col, val in new_scores.items():
                df_p4.at[idx, col] = val

            print(
                f"  ✂ {item_id} attempt {attempt}/{max_attempts}  "
                f"FK: {fk_grade:.1f} → {new_scores['flesch_kincaid_grade']:.1f}  "
                f"| {current_text[:40]}... → {new_text[:40]}"
            )

            # Update loop variables for next iteration
            fk_grade = new_scores["flesch_kincaid_grade"]
            scores   = new_scores

        except Exception as e:
            logger.warning(f"Simplification attempt {attempt} failed for {item_id}: {e}")
            break

    # Warn if still above hard cap after all attempts
    if fk_grade > P4["reading_level_hard_cap"]:
        logger.warning(
            f"  ⚠ {item_id} still above hard cap (FK: {fk_grade:.1f}) "
            f"after {attempt} attempt(s) — kept as-is for human review"
        )

    # Final readability flags based on latest scores
    df_p4.at[idx, "above_reading_target"] = fk_grade > target_grade

    # ── Bias detection — always on final item_text ────────────────────────────
    flags = detect_bias(str(df_p4.at[idx, "item_text"]))
    df_p4.at[idx, "bias_flags"]      = ", ".join(flags)
    df_p4.at[idx, "bias_flag_count"] = len(flags)

    if flags:
        print(
            f"  ⚠ {item_id} bias flags: {', '.join(flags)}  "
            f"| {str(df_p4.at[idx, 'item_text'])[:60]}"
        )

# ── Summary ───────────────────────────────────────────────────────────────────
read_summary = df_p4.groupby("scale_name").agg(
    total_items        = ("item_id",              "count"),
    mean_fk_grade      = ("flesch_kincaid_grade",  "mean"),
    items_above_target = ("above_reading_target",  "sum"),
    items_simplified   = ("was_simplified",        "sum")
).reset_index()
read_summary["mean_fk_grade"] = read_summary["mean_fk_grade"].round(2)

bias_summary_rows = []
for bt in ["gender", "age", "cultural", "socioeconomic"]:
    count = df_p4["bias_flags"].str.contains(bt, na=False).sum()
    bias_summary_rows.append({
        "bias_type":    bt,
        "items_flagged": int(count),
        "pct_of_total": round(100 * count / len(df_p4), 1) if len(df_p4) else 0
    })

print(f"\n{'='*60}")
print("Readability Summary by Scale")
print(f"{'='*60}")
print(read_summary.to_string(index=False))

print(f"\n{'='*60}")
print("Bias Summary")
print(f"{'='*60}")
print(pd.DataFrame(bias_summary_rows).to_string(index=False))


# ── System readability pass: False only if item still above hard cap after all simplification attempts
df_p4["system_readability_pass"] = ~(
    (df_p4["flesch_kincaid_grade"] > P4["reading_level_hard_cap"]) &
    (df_p4["simplification_attempts"] >= P4.get("max_simplification_attempts", 3))
)

# ── Human review columns ──────────────────────────────────────────────────────
df_p4["human_review_pass"] = True   # set False to exclude; set True to override a system fail
df_p4["human_comments"]    = ""     # e.g., "bias flag accepted", "readability acceptable for target group"

with pd.ExcelWriter(P4_OUT, engine="openpyxl") as writer:
    df_p4.to_excel(writer, sheet_name="Items_With_Readability", index=False)
    read_summary.to_excel(writer, sheet_name="Readability_Summary", index=False)
    pd.DataFrame(bias_summary_rows).to_excel(writer, sheet_name="Bias_Summary", index=False)
    if simplified_log:
        pd.DataFrame(simplified_log).to_excel(writer, sheet_name="Simplified_Items_Log", index=False)

print(f"\n✅ Phase 4 complete.")
print(f"   {int(df_p4['was_simplified'].sum())} items simplified")
print(f"   {int(df_p4['bias_flag_count'].gt(0).sum())} items with bias flags")
print(f"   {int(df_p4['above_reading_target'].sum())} items still above reading target")
print(f"📄 Output: {P4_OUT}")


# ─── PHASE 4: Content Validity – LLM SME Review ─────────────────────────────

**Input:** `03_readability_bias.xlsx` → sheet `Items_With_Readability` (respects `human_review_pass`)  
**Output:** `04_content_validity.xlsx`  
**Sheets:** `CV_All_SME_Ratings`, `CV_Item_Summary`, `CV_Scale_Summary`, `Run_Log`

---

### What this phase does

Five LLM models act as independent subject matter experts (SMEs). Each rates every item's correspondence with **every scale** in your assessment on a 1–7 Likert scale (Hinkin & Tracey, 1999). From these multi-scale ratings, two content validity indices are computed per item following the procedures in Colquitt et al. (2019) and the LM-AIG framework (Lee, Son, & Jia, 2025):

- **c-value (Correspondence)** — the average correspondence rating for the item's *intended* (target) scale, normalized to 0–1. Captures how well the item measures what it's supposed to measure.
- **d-value (Distinctiveness)** — the average of (target rating − orbiting rating) / (a − 1), where *a* is the number of anchors on the rating scale. Captures how clearly the item separates from non-target constructs.

An item **passes** if it meets both thresholds set in `pipeline_settings.json`:

- `min_c_value` ≥ 0.88
- `min_d_value` ≥ 0.35

Scale mapping agreement (the proportion of SMEs whose highest-rated scale matches the target) is retained as a diagnostic metric.

Checkpointing is applied every 100 items — if the run is interrupted, it resumes from where it left off.

> **Methodological basis:** The correspondence and distinctiveness indices were introduced by Hinkin and Tracey (1999), operationalized by Colquitt et al. (2019), and implemented in an automated LLM context by Lee, Son, and Jia (2025) in the LM-AIG multi-agent framework. Suárez-Álvarez et al. (2026) provide the broader practical guide within which these techniques operate.

### System pass/fail rules (from `pipeline_settings.json`)

| Setting | Value / Role |
|---|---|
| `min_c_value` | Default 0.88 — minimum correspondence index |
| `min_d_value` | Default 0.35 — minimum distinctiveness index |
| `min_scale_mapping_agreement` | Default 0.6 — diagnostic metric (proportion of SMEs mapping to target) |
| `sme_models` | Dict of 5 LLM models used as SMEs |
| `sme_max_workers` | Parallel threads for SME calls |

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `cv_pass` | System-set | Automated CV verdict based on c-value and d-value thresholds |
| `human_review_pass` | `True` | Override CV verdict — accept marginal items or reject borderline passers |
| `human_comments` | *(blank)* | Rationale — particularly useful when c-value or d-value is borderline |

**Phase 5 reads `human_review_pass`.** Review `04_content_validity.xlsx → CV_Item_Summary` before running Phase 5 — focus on items where c-value or d-value is just below threshold, and items where the highest-rated scale is not the target.


In [ ]:
# ==============================================================================
# PHASE 4 — Content Validity: LLM SME Review
# ==============================================================================
# Reads:  output/03_readability_bias.xlsx
# Writes: output/04_content_validity.xlsx
#   Sheets: CV_All_SME_Ratings | CV_Item_Summary | CV_Scale_Summary | Run_Log
#
# Checkpoint: saves progress every 100 items to output/04_checkpoint.json
#   - If a checkpoint exists, the run resumes from where it left off
#   - Checkpoint is deleted automatically on successful completion
#
# scale_mapping_agreement = proportion of SMEs whose mapped scale matches the
#   INTENDED/TARGET scale (not the majority vote).
# cv_majority_agreement   = proportion agreeing with the majority vote
#   (diagnostic only — kept separately).
# ==============================================================================

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

P5     = CFG["phase_04_content_validity"]
P5_IN  = BASE_OUTPUT_DIR / CFG["phase_03_readability_bias"]["output_file"]
P5_OUT = BASE_OUTPUT_DIR / P5["output_file"]
P5_CHK = BASE_OUTPUT_DIR / "04_checkpoint.json"   # checkpoint file

CHECKPOINT_INTERVAL = 100   # save every N items

assert P5_IN.exists(), f"Input not found: {P5_IN}. Run Phase 3 first."
df_p5_raw = pd.read_excel(P5_IN, sheet_name="Items_With_Readability")
# Respect human overrides from Phase 4
if "human_review_pass" in df_p5_raw.columns:
    df_p5 = df_p5_raw[df_p5_raw["human_review_pass"] == True].copy()
    n_excl = len(df_p5_raw) - len(df_p5)
    print(f"✓ Phase 3 human review: {n_excl} items excluded by human_review_pass")
else:
    df_p5 = df_p5_raw.copy()

SME_MODELS: Dict[str, Dict] = P5["sme_models"]
SME_KEYS        = list(SME_MODELS.keys())
SME_MAX_WORKERS = P5.get("sme_max_workers", len(SME_KEYS))

# ── Diagnostic: verify scales list is populated ───────────────────────────────
print(f"✓ ALL_SCALES_LIST_STR length: {len(ALL_SCALES_LIST_STR)} chars")
if not ALL_SCALES_LIST_STR:
    raise ValueError(
        "ALL_SCALES_LIST_STR is empty — re-run Section 0 to reload scales."
    )


# ── Checkpoint helpers ────────────────────────────────────────────────────────
def save_checkpoint(all_sme_rows: List[Dict], run_log: List[Dict], completed_ids: List[str]) -> None:
    payload = {
        "completed_item_ids": completed_ids,
        "all_sme_rows":       all_sme_rows,
        "run_log":            run_log,
        "saved_at":           datetime.now().isoformat()
    }
    tmp = str(P5_CHK) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f)
    os.replace(tmp, P5_CHK)   # atomic write — no corrupt checkpoint on sleep/crash
    print(f"  💾 Checkpoint saved — {len(completed_ids)} items complete")

def load_checkpoint() -> tuple:
    """Returns (all_sme_rows, run_log, completed_ids) or empty lists if no checkpoint."""
    if not P5_CHK.exists():
        return [], [], []
    with open(P5_CHK, "r", encoding="utf-8") as f:
        payload = json.load(f)
    completed_ids = payload.get("completed_item_ids", [])
    all_sme_rows  = payload.get("all_sme_rows",       [])
    run_log       = payload.get("run_log",             [])
    print(f"✓ Checkpoint found — resuming from item {len(completed_ids) + 1} "
          f"({len(completed_ids)} already complete)")
    return all_sme_rows, run_log, completed_ids


# ── Thread-safe LLM helpers ───────────────────────────────────────────────────
def make_thread_client() -> OpenAI:
    """Create a fresh OpenAI client — one per thread to avoid thread-safety issues."""
    return OpenAI(
        api_key      = OPENROUTER_API_KEY,
        base_url     = OPENROUTER_BASE_URL,
        default_headers={
            "HTTP-Referer": "https://inubilum.io",
            "X-Title":      os.getenv("OPENROUTER_APP_NAME", "Project Majdal")
        }
    )

def llm_call_threaded(
    prompt:      str,
    model:       str,
    max_tokens:  int   = 500,
    temperature: float = 0.2,
    retries:     int   = 3,
    system:      str   = "You are a psychometric expert assistant."
) -> str:
    """Thread-safe LLM call — creates its own client instance."""
    thread_client = make_thread_client()
    for attempt in range(retries):
        try:
            resp = thread_client.chat.completions.create(
                model       = model,
                messages    = [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens  = max_tokens,
                temperature = temperature
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = (2 ** attempt) + np.random.uniform(0, 1)
            logger.warning(
                f"Threaded LLM call failed (attempt {attempt+1}/{retries}): {e}. "
                f"Retrying in {wait:.1f}s"
            )
            time.sleep(wait)
    raise RuntimeError(f"Threaded LLM call failed after {retries} attempts")

def build_cv_prompt(item_text: str, scale_name: str, scale_definition: str) -> str:
    """Build a prompt asking the SME to rate the item against ALL scales on a 1-7 scale."""
    # Build a numbered scales block with full definitions for rating
    scales_block = "\n".join(
        f"- {name}: {defn[:200]}..." if len(defn) > 200 else f"- {name}: {defn}"
        for name, defn in ALL_SCALES_LOOKUP.items()
    )
    template = P5["prompt_template"]
    return template.format(
        scale_name       = scale_name,
        all_scales_block = scales_block,
        item_text        = item_text
    )

def parse_cv_response(raw: str) -> Dict:
    """Parse JSON from SME response — expects {ratings: {scale: 1-7, ...}, rationale: ...}."""
    try:
        cleaned = re.sub(r'```json|```', '', raw).strip()
        # Find the outermost JSON object (may contain nested ratings dict)
        brace_depth = 0
        start = None
        for i, ch in enumerate(cleaned):
            if ch == '{':
                if start is None:
                    start = i
                brace_depth += 1
            elif ch == '}':
                brace_depth -= 1
                if brace_depth == 0 and start is not None:
                    obj = json.loads(cleaned[start:i+1])
                    ratings = obj.get("ratings", {})
                    # Normalize scale names (strip whitespace)
                    ratings = {k.strip(): float(v) for k, v in ratings.items()}
                    return {
                        "scale_ratings": ratings,
                        "rationale":     str(obj.get("rationale", ""))
                    }
    except Exception:
        pass
    return {
        "scale_ratings": {},
        "rationale":     raw[:200]
    }

def review_item_with_sme(
    item_text:  str,
    scale_name: str,
    scale_def:  str,
    sme_key:    str
) -> Dict:
    """Call a single SME model with its own thread-safe client."""
    model_id = SME_MODELS[sme_key]["model_id"]
    prompt   = build_cv_prompt(item_text, scale_name, scale_def)
    try:
        raw    = llm_call_threaded(
            prompt, model_id,
            max_tokens  = P5["max_tokens"],
            temperature = P5["temperature"],
            retries     = P5["retry_attempts"],
            system      = "You are a psychometric subject matter expert. Respond only in valid JSON."
        )
        result = parse_cv_response(raw)
        result["sme"]          = sme_key
        result["raw_response"] = raw[:400]
        result["error"]        = ""
    except Exception as e:
        result = {
            "sme":               sme_key,
            "scale_ratings":     {},
            "rationale":         "",
            "raw_response":      "",
            "error":             str(e)
        }
    return result


# ── Quick single-call diagnostic before full run ──────────────────────────────
print("\nRunning single-call diagnostic...")
_test = review_item_with_sme(
    item_text  = "I enjoy solving complex problems",
    scale_name = "Abstract Thinking",
    scale_def  = "The ability to think conceptually and recognize patterns",
    sme_key    = SME_KEYS[0]
)
if _test["error"]:
    raise RuntimeError(
        f"Diagnostic call failed for {SME_KEYS[0]}: {_test['error']}\n"
        "Fix this before running the full phase."
    )
print(f"✓ Diagnostic passed — {SME_KEYS[0]} returned rating: {_test['target_fit_rating']}")


# ── Load checkpoint if available ──────────────────────────────────────────────
all_sme_rows, run_log_p5, completed_ids = load_checkpoint()
completed_set = set(completed_ids)

# Filter df_p5 to only unprocessed items
df_remaining = df_p5[~df_p5["item_id"].isin(completed_set)].copy()

print(f"\n{'='*60}")
print(f"PHASE 4: Content Validity – LLM SME Review")
print(f"Total items     : {len(df_p5)}")
print(f"Already done    : {len(completed_set)}")
print(f"Remaining       : {len(df_remaining)}")
print(f"SMEs            : {len(SME_KEYS)}")
print(f"Parallel workers: {SME_MAX_WORKERS}")
print(f"Remaining calls : {len(df_remaining) * len(SME_KEYS)}")
print(f"SME models      : {', '.join(SME_KEYS)}")
print(f"Pass threshold  : c-value >= {MIN_C_VALUE} AND d-value >= {MIN_D_VALUE}")
print(f"Checkpoint every: {CHECKPOINT_INTERVAL} items  →  {P5_CHK.name}")
print(f"{'='*60}\n")

# ── Main processing loop ──────────────────────────────────────────────────────
items_since_checkpoint = 0

for idx, row in tqdm(df_remaining.iterrows(), total=len(df_remaining), desc="Items"):
    item_id    = row.get("item_id",    f"item_{idx}")
    item_text  = str(row.get("item_text",  ""))
    scale_name = str(row.get("scale_name", ""))
    scale      = scale_params_lookup.get(scale_name, {})
    scale_def  = scale.get("construct_definition", ALL_SCALES_LOOKUP.get(scale_name, ""))

    item_ratings:  Dict[str, float] = {}
    item_mappings: Dict[str, str]   = {}
    item_all_ratings: Dict[str, Dict[str, float]] = {}
    t0_item = time.time()

    # ── Run all SMEs in parallel ──────────────────────────────────────────────
    with ThreadPoolExecutor(max_workers=SME_MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                review_item_with_sme, item_text, scale_name, scale_def, sme_key
            ): sme_key
            for sme_key in SME_KEYS
        }
        for future in as_completed(futures):
            sme_key = futures[future]
            try:
                result = future.result()
            except Exception as e:
                result = {
                    "sme":               sme_key,
                    "scale_ratings":     {},
                    "rationale":         "",
                    "raw_response":      "",
                    "error":             str(e)
                }
            result.update({
                "item_id":               item_id,
                "scale_name":            scale_name,
                "item_text":             item_text,
                "source_example":        row.get("source_example",        ""),
                "occurrence_likelihood": row.get("occurrence_likelihood",  ""),
                "difficulty_target":     row.get("difficulty_target",      ""),
                "flesch_kincaid_grade":  row.get("flesch_kincaid_grade",   0),
                "was_simplified":        row.get("was_simplified",         False),
                "bias_flags":            row.get("bias_flags",             ""),
                "duration_s":            round(time.time() - t0_item, 2)
            })
            all_sme_rows.append(result)
            scale_ratings = result.get("scale_ratings", {})
            # Find target rating and best-rated scale for this SME
            target_rating = scale_ratings.get(scale_name, 0.0)
            item_ratings[sme_key]  = target_rating
            best_scale = max(scale_ratings, key=scale_ratings.get) if scale_ratings else "unknown"
            item_mappings[sme_key] = best_scale
            item_all_ratings[sme_key] = scale_ratings

    # Per-item summary line
    ratings_str = "  ".join(f"{k}: {v:.1f}" for k, v in item_ratings.items())
    mean_rating = round(float(np.mean(list(item_ratings.values()))), 2) if item_ratings else 0.0
    print(
        f"  {item_id}  |  target_mean: {mean_rating}  |  "
        f"{ratings_str}  |  best: {list(item_mappings.values())}"
    )

    run_log_p5.append({
        "item_id":     item_id,
        "scale_name":  scale_name,
        "sme_count":   len(SME_KEYS),
        "mean_rating": mean_rating,
        "duration_s":  round(time.time() - t0_item, 2),
        "timestamp":   datetime.now().isoformat()
    })

    completed_ids.append(item_id)
    items_since_checkpoint += 1

    # ── Checkpoint every N items ──────────────────────────────────────────────
    if items_since_checkpoint >= CHECKPOINT_INTERVAL:
        save_checkpoint(all_sme_rows, run_log_p5, completed_ids)
        items_since_checkpoint = 0

    time.sleep(P5["inter_item_delay_seconds"])

# Serialize scale_ratings dicts for Excel storage
for row in all_sme_rows:
    sr = row.get("scale_ratings", {})
    if isinstance(sr, dict):
        row["scale_ratings_str"] = str(sr)
    else:
        row["scale_ratings_str"] = str(sr)

df_sme_raw = pd.DataFrame(all_sme_rows)

# ── Compute agreement metrics per item ────────────────────────────────────────
print(f"\n{'='*60}")
print("Computing agreement metrics...")
print(f"{'='*60}")

item_summary_rows: List[Dict] = []

# Number of anchors on the rating scale (1-7 Likert)
N_ANCHORS = 7
MIN_C_VALUE = P5["pass_thresholds"]["min_c_value"]
MIN_D_VALUE = P5["pass_thresholds"]["min_d_value"]
MIN_MAPPING = P5["pass_thresholds"].get("min_scale_mapping_agreement", 0.6)

for item_id, grp in df_sme_raw.groupby("item_id"):
    first_row    = grp.iloc[0]
    target_scale = first_row["scale_name"]

    # Collect per-SME scale ratings
    sme_scale_ratings: List[Dict[str, float]] = []
    for _, sme_row in grp.iterrows():
        sr = sme_row.get("scale_ratings", {})
        if isinstance(sr, str):
            try:
                sr = json.loads(sr.replace("'", '"'))
            except Exception:
                sr = {}
        if sr:
            sme_scale_ratings.append(sr)

    # ── Compute c-value and d-value (Colquitt et al., 2019) ──────────────
    if sme_scale_ratings:
        # c-value: average target rating normalized to 0-1
        target_ratings = [sr.get(target_scale, 0.0) for sr in sme_scale_ratings]
        mean_target_raw = float(np.mean(target_ratings))
        c_value = round(mean_target_raw / N_ANCHORS, 3)

        # d-value: average of (target - orbiting) / (a - 1) across all orbiting scales
        orbiting_scales = [s for s in ALL_SCALES_LOOKUP.keys() if s != target_scale]
        if orbiting_scales:
            d_values_per_sme = []
            for sr in sme_scale_ratings:
                t_rating = sr.get(target_scale, 0.0)
                orb_diffs = [(t_rating - sr.get(os, 0.0)) for os in orbiting_scales]
                mean_diff = float(np.mean(orb_diffs))
                d_values_per_sme.append(mean_diff / (N_ANCHORS - 1))
            d_value = round(float(np.mean(d_values_per_sme)), 3)
        else:
            d_value = 1.0  # only one scale — trivially distinct

        mean_target_rating = round(mean_target_raw, 3)
        std_target_rating  = round(float(np.std(target_ratings)), 3)
        min_rating         = min(target_ratings)
        max_rating         = max(target_ratings)
    else:
        c_value = 0.0
        d_value = 0.0
        mean_target_rating = 0.0
        std_target_rating  = 0.0
        min_rating = 0
        max_rating = 0

    # ── Scale mapping: which scale each SME rated highest ────────────────
    mapped_scales = []
    for sr in sme_scale_ratings:
        if sr:
            best = max(sr, key=sr.get)
            mapped_scales.append(best)
        else:
            mapped_scales.append("unknown")

    valid_scales = [s for s in mapped_scales if s != "unknown"]
    target_count = valid_scales.count(target_scale)
    scale_mapping_agreement = round(target_count / len(SME_KEYS), 3) if SME_KEYS else 0.0

    if valid_scales:
        most_common_scale, most_common_count = Counter(valid_scales).most_common(1)[0]
        majority_agreement = round(most_common_count / len(SME_KEYS), 3)
    else:
        most_common_scale  = "unknown"
        majority_agreement = 0.0

    # ── CV pass: both c-value and d-value must meet thresholds ───────────
    cv_pass = (c_value >= MIN_C_VALUE and d_value >= MIN_D_VALUE)

    item_summary_rows.append({
        "item_id":                    item_id,
        "scale_name":                 target_scale,
        "item_text":                  first_row["item_text"],
        "source_example":             first_row.get("source_example",        ""),
        "occurrence_likelihood":      first_row.get("occurrence_likelihood",  ""),
        "difficulty_target":          first_row.get("difficulty_target",      ""),
        "flesch_kincaid_grade":       first_row.get("flesch_kincaid_grade",   0),
        "was_simplified":             first_row.get("was_simplified",         False),
        "bias_flags":                 first_row.get("bias_flags",             ""),
        "cv_c_value":                 c_value,
        "cv_d_value":                 d_value,
        "cv_mean_target_rating":      mean_target_rating,
        "cv_std_target_rating":       std_target_rating,
        "cv_min_rating":              min_rating,
        "cv_max_rating":              max_rating,
        "cv_scale_mapping_agreement": scale_mapping_agreement,
        "cv_maps_to_target_count":    target_count,
        "cv_maps_to_target":          target_count == len(SME_KEYS),
        "cv_majority_mapped_scale":   most_common_scale,
        "cv_majority_agreement":      majority_agreement,
        "cv_majority_is_target":      most_common_scale == target_scale,
        "cv_sme_scale_ratings":       str({
                                          sme: sr
                                          for sme, sr in zip(grp["sme"], sme_scale_ratings)
                                      }) if sme_scale_ratings else "{}",
        "cv_pass":                    cv_pass
    })

df_cv_summary = pd.DataFrame(item_summary_rows)

# ── Scale-level CV summary ────────────────────────────────────────────────────
cv_scale_summary = df_cv_summary.groupby("scale_name").agg(
    total_items            = ("item_id",                   "count"),
    cv_pass_count          = ("cv_pass",                   "sum"),
    mean_c_value           = ("cv_c_value",                 "mean"),
    mean_d_value           = ("cv_d_value",                 "mean"),
    mean_target_rating     = ("cv_mean_target_rating",      "mean"),
    mean_agreement         = ("cv_scale_mapping_agreement", "mean"),
    unanimous_target_pct   = ("cv_maps_to_target",          "mean"),
    majority_is_target_pct = ("cv_majority_is_target",      "mean")
).reset_index()

cv_scale_summary["cv_pass_rate_pct"]        = round(
    100 * cv_scale_summary["cv_pass_count"] / cv_scale_summary["total_items"], 1
)
cv_scale_summary["mean_target_rating"]      = cv_scale_summary["mean_target_rating"].round(2)
cv_scale_summary["mean_agreement"]          = cv_scale_summary["mean_agreement"].round(2)
cv_scale_summary["unanimous_target_pct"]    = (cv_scale_summary["unanimous_target_pct"] * 100).round(1)
cv_scale_summary["majority_is_target_pct"]  = (cv_scale_summary["majority_is_target_pct"] * 100).round(1)

print(f"\n{'='*60}")
print("CV Scale Summary")
print(f"{'='*60}")
cv_scale_summary["mean_c_value"] = cv_scale_summary["mean_c_value"].round(3)
cv_scale_summary["mean_d_value"] = cv_scale_summary["mean_d_value"].round(3)

print(cv_scale_summary[[
    "scale_name", "total_items", "cv_pass_count",
    "cv_pass_rate_pct", "mean_c_value", "mean_d_value", "mean_target_rating", "mean_agreement"
]].to_string(index=False))

# ── Write Excel ───────────────────────────────────────────────────────────────

# ── Human review columns added to item summary ────────────────────────────────
# cv_pass is the system verdict; human_review_pass lets reviewers override it
df_cv_summary["human_review_pass"] = True   # set False to exclude; True to reinstate
df_cv_summary["human_comments"]    = ""

with pd.ExcelWriter(P5_OUT, engine="openpyxl") as writer:
    df_sme_raw      .to_excel(writer, sheet_name="CV_All_SME_Ratings", index=False)
    df_cv_summary   .to_excel(writer, sheet_name="CV_Item_Summary",    index=False)
    cv_scale_summary.to_excel(writer, sheet_name="CV_Scale_Summary",   index=False)
    pd.DataFrame(run_log_p5).to_excel(writer, sheet_name="Run_Log",    index=False)

# ── Delete checkpoint on successful completion ────────────────────────────────
if P5_CHK.exists():
    P5_CHK.unlink()
    print("🗑️  Checkpoint deleted — run completed successfully.")

cv_pass_total = df_cv_summary["cv_pass"].sum()
print(f"\n✅ Phase 4 complete.")
print(f"   {int(cv_pass_total)}/{len(df_cv_summary)} items passed CV thresholds (c≥{MIN_C_VALUE}, d≥{MIN_D_VALUE}).")
print(f"📄 Output: {P5_OUT}")


# ─── PHASE 5: Pseudo-Factor Analysis and Final Review ────────────────────────────────────────

**Input:** `04_content_validity.xlsx` → sheet `CV_Item_Summary` (respects `human_review_pass`)  
**Output:** `05_pseudo_factor_analysis.xlsx`  
**Sheets:** `PFA_Item_Results`, `PFA_Scale_Summary`, `Discriminant_Matrix`, `Item_Scale_Heatmap`, `Loading_Matrix_All`, `Loading_Matrix_CV_Pass`, `Fit_Statistics`, `Eigenvalues`, `Tucker_Congruence`, `High_Pass_Loadings`, `All_Pass_Loadings`, `Pass_Rules_Definition`

---

### What this phase does

Phase 5 is the statistical core of the pipeline. Using sentence-transformer embeddings as a semantic proxy for item inter-correlations, it applies factor analysis to evaluate each item's psychometric properties *before* collecting any real human response data — the Pseudo-Factor Analysis (PFA) technique introduced by Guenole et al. (2025) and independently evaluated by Suárez-Álvarez et al. (2026).

**Methodological basis:**

> Guenole et al. (2025) introduced PFA and demonstrated that the *substitutability assumption* — that embedding similarity can approximate empirical item correlations — holds well enough for early-stage item selection across two well-established personality frameworks (NEO and HEXACO). Suárez-Álvarez et al. (2026) conducted the first independent evaluation of PFA, confirming its utility while identifying boundary conditions. Milano et al. (2025) provide converging evidence that LLM-derived embeddings capture a-priori factorial structure with moderate-to-high correspondence to human-response-based factor solutions across four validated personality tests.

---

### Key design decisions

**1. Multi-transformer aggregation (Guenole et al., 2025)**

One or more sentence transformer models can be specified in `pipeline_settings.json` under `transformer_models`. If multiple models are listed, their cosine similarity matrices are averaged before factor analysis — Guenole et al. (2025) found that aggregated multi-transformer matrices consistently outperform any single transformer in factor structure recovery. The default is `all-MiniLM-L6-v2` only. To use the dual-model approach from the original Guenole et al. (2025) design, add `all-mpnet-base-v2` to the list. The `Fit_Statistics` sheet records which models were used. The heatmap and projection calculations use the first model in the list (the primary model) only, to ensure consistent embedding dimensionality.

**2. Negative item detection and reversal for embedding**

Items that are semantically low-pole — describing the absence or avoidance of the target construct (e.g., "I avoid planning ahead") — are detected via a two-stage process: a regex heuristic for avoidance language, followed by an LLM call for borderline cases. Detected items are rewritten to their positive pole *for embedding only*. The original `item_text` is always preserved in all output. Reversed items are flagged in the `item_reversed_for_embedding` column. This follows the atomic-reversed encoding logic evaluated in Guenole et al. (2025).

**3. Dual encoding (atomic + macro)**

Both encoding strategies are run and their results reported per item. Atomic encoding embeds each item individually and runs factor analysis on the aggregated similarity matrix. Macro encoding concatenates all items in a scale and computes item-to-scale cosine similarity. Guenole et al. (2025) found that neither strategy dominates universally — macro performed best for NEO while atomic performed best for HEXACO — supporting the value of reporting both.

**4. Tucker's congruence coefficient (Lorenzo-Seva & ten Berge, 2006)**

Tucker's congruence coefficient is computed between the atomic single-factor loading vector and the macro item-to-scale similarity vector for each scale. Values ≥ 0.95 indicate excellent encoding invariance; ≥ 0.85 indicates fair similarity; < 0.85 suggests the two strategies disagree meaningfully and warrants closer inspection of the construct definition. Results appear in the `Tucker_Congruence` sheet and are summarised in `PFA_Scale_Summary`. The post-trial analogue — congruence between PFA loadings and empirical loadings from real human data — is the gold-standard check recommended by Guenole et al. (2025) and should be computed once trial data are available.

**5. DAAL factor identity**

In the joint multi-scale factor analysis, each factor is labelled using the **Dominant Average Absolute Loading (DAAL)** approach (Guenole et al., 2025). DAAL works as follows: for each extracted factor, the mean of the absolute loadings of every item belonging to a given scale is computed — this is the *average absolute loading* for that scale on that factor. This is done for every scale on every factor. The scale with the *dominant* (highest) average absolute loading on a given factor is then assigned as that factor's label. A scale's factor identity is confirmed if the factor labelled with its name is also the factor on which that scale's items load most strongly — in other words, its items are dominated by their own factor rather than another scale's factor. Where two or more scales produce the same dominant average absolute loading on the same factor, the label cannot be uniquely assigned and the factor is marked `Unassigned_F{n}`. These unassigned factors still appear in the `Discriminant_Matrix` for completeness but do not contribute to DAAL identity confirmation. DAAL results are reported as flags for human review — not as hard pass/fail criteria — because heterogeneous scale frameworks may produce uninterpretable joint solutions.

**6. Model fit — RMSR only**

RMSR (Root Mean Square Residual) is the sole reported fit metric. CFI, TLI, and RMSEA are not computed — they require a known sample size and are unreliable in PFA contexts (Guenole et al., 2025; Suárez-Álvarez et al., 2026). Values below 0.08 indicate acceptable fit; values around 0.03–0.05 indicate good fit.

---

### Output sheets — what each one contains and when to use it

**`PFA_Item_Results`** is the primary per-item results sheet. Every item that entered PFA has its full set of metrics here — atomic loadings, macro similarity, pass rules, discriminant validity flags, and DAAL identity. This is the sheet to use when making item-level decisions.

**`PFA_Scale_Summary`** provides one row per scale with aggregated statistics, DAAL identity, Tucker's congruence, and discriminant flag counts. Start here for a scale-level overview before drilling into individual items.

**`Discriminant_Matrix`** contains the raw FA loadings from the joint multi-scale analysis — one row per item, one column per extracted factor in rotation order. Columns are labelled with the scale name where DAAL resolved cleanly, or `Unassigned_F{n}` where it did not. This is the most psychometrically rigorous view of inter-scale relationships, but it only covers scales where the joint FA ran successfully, and the column order follows FA rotation rather than your scale definitions. Use this sheet when you want to examine the precise factor-analytic structure or investigate a specific cross-loading value.

**`Item_Scale_Heatmap`** is the most human-readable inter-scale view. It contains one row per item and one column per scale in `scales.json` order, with values representing cosine similarity between each item's embedding and each scale's macro embedding. No factor analysis or DAAL is involved — every item gets a value against every scale regardless of whether DAAL resolved that scale. Values typically range from roughly 0.3 to 0.8. Scanning across a row, the highest value should fall on the item's own scale column; items where the peak falls elsewhere are discriminant validity concerns. Use this sheet for visual inspection and for reviewing scales that DAAL could not assign — it provides complete coverage where the Discriminant_Matrix has gaps.

**`Loading_Matrix_All`** restructures the joint FA loadings as an item × scale matrix (rather than item × factor), covering all 600 Phase 5 items including those that failed CV. Items that did not enter the PFA solution have *projected* loadings estimated via similarity-weighted projection onto the joint factor space — these are approximations and are flagged `in_pfa_solution = False`. Use this sheet when you need FA-based loading values for items across the full Phase 5 pool.

**`Loading_Matrix_CV_Pass`** is identical in structure to `Loading_Matrix_All` but contains only items that passed CV. All loadings here are exact FA values with no projections. Use this sheet in preference to `Loading_Matrix_All` when you want to work only with items that met the content validity threshold.

**`Fit_Statistics`** reports RMSR and residual distribution statistics per scale, along with the transformer models used. Check `fit_acceptable` here before interpreting item-level results for any scale.

**`Tucker_Congruence`** reports the congruence coefficient between atomic and macro solutions for each scale, with an interpretation label. Scales with poor congruence (< 0.85) should be examined closely in `Item_Scale_Heatmap` — divergence between encoding strategies is often a signal of construct definition ambiguity.

**`High_Pass_Loadings`** and **`All_Pass_Loadings`** are filtered subsets of `PFA_Item_Results` containing only HIGH_PASS items and all trial-ready items respectively, with a reduced column set for easier review.

---

### Six pass/fail rules (configurable in `pipeline_settings.json`)

| Rule | Criterion | Default threshold |
|---|---|---|
| Rule 1 – Strong Single Factor | Atomic single-factor loading ≥ threshold | 0.40 |
| Rule 2 – Adequate Single Factor | Atomic single-factor loading ≥ threshold | 0.30 |
| Rule 3 – Adequate Communality | Communality ≥ threshold | 0.15 |
| Rule 4 – Clean Factor Structure | Max secondary loading < threshold | 0.35 |
| Rule 5 – Cross-Difficulty Viable | All-items loading ≥ threshold | 0.20 |
| Rule 6 – Within-Difficulty Viable | Within-difficulty loading ≥ threshold | 0.25 |

An item is **HIGH_PASS** if it meets Rules 1, 3, and 4. It is **STANDARD_PASS** if it meets at least `min_rules_to_pass` rules (default 3). Items below this threshold are excluded.

---

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `item_reversed_for_embedding` | System-set | `True` if item was rewritten to positive pole for embedding; original `item_text` always preserved |
| `pfa_pass_level` | System-set | `HIGH_PASS`, `STANDARD_PASS`, or `FAIL` |
| `human_review_pass` | `True` | Override PFA verdict — set `False` to exclude a passing item; set `True` to reinstate a failed one |
| `human_comments` | *(blank)* | Notes — especially important for DAAL failures, discriminant flags, and reversed items |

**Review the output after this phase completes.** Review `05_pseudo_factor_analysis.xlsx` completing review. A suggested review sequence:

1. `Tucker_Congruence` — identify scales with poor congruence (< 0.85) for closer attention
2. `PFA_Scale_Summary` — scales where `daal_identity_confirmed = False` and high `disc_flag_count`
3. `Item_Scale_Heatmap` — visual scan for items whose peak similarity falls on the wrong scale column
4. `PFA_Item_Results` — item-level decisions; filter on `discriminant_flag_review = True` and `item_reversed_for_embedding = True` first

---

### Final Review: Item Pool Assembly

After PFA completes, Phase 5 assembles the final human-review-ready item pool. Only items that passed **both** Phase 4 (Content Validity) and PFA — as determined by the `human_review_pass` columns from each — are included in `Review_Pool`.

**Additional output sheets:**

- **`Review_Pool`** — all passing items with full quality metadata. Two blank columns for reviewer decisions.
- **`Rejected_Items`** — excluded items, preserved for transparency.
- **`Pool_Statistics`** — one row per scale: item counts by difficulty, mean CV rating, mean atomic loading, bias flag counts, reversal counts.
- **`Quality_Flags`** — every item with at least one automated concern: readability, bias, low CV rating, mapping failure, standard pass only, discriminant flag, DAAL failure, or reversal. Review these first.
- **`Recommendations`** — scale-level guidance with HIGH / MEDIUM priority: scales with too few items, high mean FK grade, multiple discriminant flags, DAAL failures, and reversal counts.

**Reviewer workflow:**

1. Open `05_pseudo_factor_analysis.xlsx`
2. Start with `Recommendations` and `Quality_Flags` sheets
3. Work through `Review_Pool` scale by scale
4. Record decisions in `human_decision` and rationale in `human_notes`
5. Items marked `reject` or `modify` can be flagged for regeneration by re-running earlier phases with adjusted settings

> **This is a pre-trial item pool.** Real human response data and full psychometric validation are required before any items are used operationally. The pipeline replaces weeks of manual drafting and initial screening — it does not replace empirical validation (Guenole et al., 2025; Suárez-Álvarez et al., 2026).


In [ ]:
# ==============================================================================
# PHASE 5 — Pseudo-Factor Analysis (PFA)
# ==============================================================================
# Reads:  output/04_content_validity.xlsx  (CV_Item_Summary sheet)
#         output/03_readability_bias.xlsx   (Items_With_Readability sheet)
# Writes: output/05_pseudo_factor_analysis.xlsx
#
# Sheets:
#   PFA_Item_Results        — per-item PFA metrics for both encoding strategies
#   PFA_Scale_Summary       — scale-level summary with DAAL identity, discriminant,
#                             and Tucker's congruence
#   Discriminant_Matrix     — full item × factor loading matrix (all factors,
#                             all items, factor-index order including Unassigned)
#   Item_Scale_Heatmap      — item × scale cosine similarity matrix in scales.json
#                             order; every item against every scale, no DAAL dependency
#   Loading_Matrix_All      — item × scale loading matrix, ALL Phase 5 items
#   Loading_Matrix_CV_Pass  — item × scale loading matrix, cv_pass items only
#   Fit_Statistics          — RMSR + residual stats per scale
#   Eigenvalues             — eigenvalue tables per scale
#   Tucker_Congruence       — Tucker's CC between atomic and macro per scale
#   High_Pass_Loadings      — items meeting HIGH_PASS criteria
#   All_Pass_Loadings       — all items ready for trial
#   Pass_Rules_Definition   — threshold documentation
#
# KEY DESIGN DECISIONS:
#   1. Multi-transformer aggregation — similarity matrices averaged across
#      configured transformer models before FA (Guenole et al. 2025)
#   2. Negative item detection & reversal for embedding — low-pole items detected
#      via heuristic + LLM call, rewritten for embedding only, flagged in output
#   3. Tucker's congruence — atomic vs macro per scale (Lorenzo-Seva & ten Berge 2006)
#   4. DAAL factor identity (Guenole et al. 2025)
#   5. RMSR only — CFI/TLI/RMSEA not computed (require sample size)
#   6. Item_Scale_Heatmap uses primary model (first in list) for both item
#      and scale embeddings to ensure consistent dimensionality
# ==============================================================================

# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

import re as _re
from sentence_transformers import SentenceTransformer
from factor_analyzer import FactorAnalyzer

P6     = CFG["phase_05_pseudo_factor_analysis"]
P6_IN  = BASE_OUTPUT_DIR / CFG["phase_04_content_validity"]["output_file"]
P6_OUT = BASE_OUTPUT_DIR / P6["output_file"]

assert P6_IN.exists(), f"Input not found: {P6_IN}. Run Phase 4 first."

# ── Settings ──────────────────────────────────────────────────────────────────
PASS_RULES_CFG    = P6["pass_rules"]
MIN_RULES         = P6["min_rules_to_pass"]
RMSR_THRESHOLD    = P6["model_fit_thresholds"]["rmsr_max"]
DISC_RATIO_MIN    = P6.get("discriminant_validity", {}).get("target_to_max_ratio_min", 1.5)
DISC_CEILING      = P6.get("discriminant_validity", {}).get("max_cross_loading_ceiling", 0.45)
DISC_CEIL_ENABLED = P6.get("discriminant_validity", {}).get("ceiling_enabled", True)
MIN_ITEMS_PFA     = P6.get("min_items_for_pfa", 4)
MIN_SCALES_JOINT  = P6.get("min_scales_for_joint_analysis", 3)
TRANSFORMER_MODELS = P6.get("transformer_models", [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2"
])

# ── Load Phase 5 data ─────────────────────────────────────────────────────────
df_cv_all = pd.read_excel(P6_IN, sheet_name="CV_Item_Summary")
if "human_review_pass" in df_cv_all.columns:
    df_cv = df_cv_all[df_cv_all["human_review_pass"] != False].copy().reset_index(drop=True)
    n_excluded = len(df_cv_all) - len(df_cv)
    print(f"✓ Phase 4 human review: {n_excluded} items excluded by human_review_pass")
else:
    df_cv = df_cv_all[df_cv_all["cv_pass"] == True].copy().reset_index(drop=True)
    n_excluded = len(df_cv_all) - len(df_cv)

print(f"✓ Phase 4 items loaded : {len(df_cv_all)}")
print(f"  Entering PFA         : {len(df_cv)}")
print(f"  Excluded             : {n_excluded}")

# ── Merge Phase 4 readability columns ─────────────────────────────────────────
df_read = pd.read_excel(
    BASE_OUTPUT_DIR / CFG["phase_03_readability_bias"]["output_file"],
    sheet_name="Items_With_Readability"
)[[
    "item_id", "flesch_kincaid_grade", "bias_flags",
    "bias_flag_count", "was_simplified", "simplification_attempts"
]].copy()

df_p6 = df_cv.merge(df_read, on="item_id", how="left")

# ── Load transformer models ───────────────────────────────────────────────────
print(f"\nLoading {len(TRANSFORMER_MODELS)} sentence transformer(s)...")
st_models = []
for model_name in TRANSFORMER_MODELS:
    print(f"  Loading: {model_name}")
    st_models.append(SentenceTransformer(model_name))
print(f"✓ {len(st_models)} transformer(s) loaded")

# Primary model (index 0) is used wherever a single consistent embedding space
# is needed (heatmap, projection). Must always be first in TRANSFORMER_MODELS.
PRIMARY_MODEL = st_models[0]


# ==============================================================================
# NEGATIVE ITEM DETECTION & REVERSAL FOR EMBEDDING
# ==============================================================================
_reversal_cache: Dict[str, str]  = {}
_reversed_flags: Dict[str, bool] = {}

def detect_and_reverse_item(item_id: str, item_text: str, scale_name: str) -> tuple:
    """
    Returns (embedding_text, was_reversed).
    Original item_text is always preserved in all output columns.
    """
    if item_id in _reversal_cache:
        return _reversal_cache[item_id], _reversed_flags[item_id]

    avoidance_patterns = [
        r'\b(avoid|never|rarely|seldom|struggle|fail|lack|dislike|hate|'
        r'reluctant|refuse|neglect|ignore|skip|miss|forget|lose|give up|'
        r'put off|procrastinat)\b'
    ]
    needs_check = any(_re.search(p, item_text.lower()) for p in avoidance_patterns)

    if not needs_check:
        _reversal_cache[item_id] = item_text
        _reversed_flags[item_id] = False
        return item_text, False

    reversal_cfg = P6.get("reversal_detection", {})
    prompt = f"""You are reviewing a personality survey item for embedding purposes.

Item: "{item_text}"
Construct: {scale_name}

Is this item semantically LOW-POLE — i.e., does it describe the ABSENCE, AVOIDANCE,
or LOW expression of the construct, even if it does not use "don't" or "never"?

If YES: rewrite it as a HIGH-POLE item with equivalent meaning for embedding only.
The rewrite should be a natural "I" statement in present tense, max 9 words.
If NO: return the original item unchanged.

Respond in JSON only:
{{"is_low_pole": true/false, "embedding_text": "<rewritten or original item>"}}"""

    try:
        raw = llm_call(
            prompt,
            model       = reversal_cfg.get("model", CFG["phase_01_behavioral_domains"]["model"]),
            max_tokens  = reversal_cfg.get("max_tokens", 100),
            temperature = reversal_cfg.get("temperature", 0.0)
        )
        cleaned = _re.sub(r'```json|```', '', raw).strip()
        # Extract only the first complete JSON object — discard any trailing text
        match = _re.search(r'\{.*?\}', cleaned, _re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in response")
        result = json.loads(match.group())
        
        is_low  = bool(result.get("is_low_pole", False))
        emb_txt = str(result.get("embedding_text", item_text)).strip()
        if not emb_txt.startswith("I "):
            emb_txt = item_text
    except Exception as e:
        logger.warning(f"Reversal check failed for {item_id}: {e} — using original")
        is_low  = False
        emb_txt = item_text

    _reversal_cache[item_id] = emb_txt
    _reversed_flags[item_id] = is_low
    return emb_txt, is_low


def get_embedding_texts(items_df: pd.DataFrame, scale_name: str) -> tuple:
    emb_texts, rev_flags = [], []
    for _, row in items_df.iterrows():
        txt, rev = detect_and_reverse_item(
            row["item_id"], row["item_text"], scale_name
        )
        emb_texts.append(txt)
        rev_flags.append(rev)
    return emb_texts, rev_flags


# ==============================================================================
# PREPROCESSING
# ==============================================================================
def preprocess_for_embedding(text: str) -> str:
    t = str(text).strip().lower()
    t = _re.sub(r'^i\s+', '', t)
    t = _re.sub(r'[.!?,;:]+$', '', t)
    t = _re.sub(r'\s+', ' ', t).strip()
    return t


# ==============================================================================
# ENCODING
# ==============================================================================
def encode_atomic_aggregate(texts: List[str]) -> np.ndarray:
    """Aggregate cosine similarity matrix averaged across all transformers."""
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    sim_matrices = []
    for model in st_models:
        embs = model.encode(preprocessed, show_progress_bar=False)
        sim  = np.array(sk_cosine(embs))
        np.fill_diagonal(sim, 1.0)
        sim_matrices.append(sim)
    agg = np.mean(sim_matrices, axis=0)
    np.fill_diagonal(agg, 1.0)
    return agg

def encode_atomic_raw(texts: List[str]) -> np.ndarray:
    """Raw embeddings from primary model only — used for projection and heatmap."""
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    return PRIMARY_MODEL.encode(preprocessed, show_progress_bar=False)

def encode_macro_aggregate(texts: List[str]) -> np.ndarray:
    """
    Item-to-scale cosine similarities averaged across all transformers.
    Each model computes item embeddings and a scale embedding from the
    concatenated text, then cosine similarities are averaged.
    Returns shape: (n_items,)
    """
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    concatenated  = " ".join(preprocessed)
    sims_per_model = []
    for model in st_models:
        scale_vec  = model.encode([concatenated],  show_progress_bar=False)[0]
        item_vecs  = model.encode(preprocessed,    show_progress_bar=False)
        scale_norm = scale_vec  / (np.linalg.norm(scale_vec)                        + 1e-9)
        items_norm = item_vecs  / (np.linalg.norm(item_vecs, axis=1, keepdims=True) + 1e-9)
        sims_per_model.append(items_norm @ scale_norm)
    return np.mean(sims_per_model, axis=0)

def encode_macro_primary(texts: List[str]) -> np.ndarray:
    """
    Scale macro embedding using primary model only.
    Returns a single vector of shape (primary_model_dim,).
    Used for heatmap — must match encode_atomic_raw dimensionality.
    """
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    concatenated  = " ".join(preprocessed)
    return PRIMARY_MODEL.encode([concatenated], show_progress_bar=False)[0]

def embeddings_to_sim_matrix(embeddings: np.ndarray) -> np.ndarray:
    sim = np.array(sk_cosine(embeddings))
    np.fill_diagonal(sim, 1.0)
    return sim


# ==============================================================================
# TUCKER'S CONGRUENCE
# ==============================================================================
def tucker_congruence(loadings_a: np.ndarray, loadings_b: np.ndarray) -> List[float]:
    """
    Tucker's congruence coefficient between corresponding columns.
    >= 0.95 excellent, >= 0.85 fair, < 0.85 poor (Lorenzo-Seva & ten Berge, 2006).
    """
    if loadings_a is None or loadings_b is None:
        return []
    if loadings_a.shape != loadings_b.shape:
        min_f      = min(loadings_a.shape[1], loadings_b.shape[1])
        loadings_a = loadings_a[:, :min_f]
        loadings_b = loadings_b[:, :min_f]
    congruence = []
    for f in range(loadings_a.shape[1]):
        a = loadings_a[:, f]
        b = loadings_b[:, f]
        num   = float(np.sum(a * b))
        denom = float(np.sqrt(np.sum(a**2) * np.sum(b**2)))
        congruence.append(round(num / denom, 4) if denom > 0 else 0.0)
    return congruence


# ==============================================================================
# FACTOR ANALYSIS
# ==============================================================================
def run_fa(sim_matrix: np.ndarray, n_factors: int):
    n = sim_matrix.shape[0]
    if n < n_factors + 2:
        return None, None
    try:
        fa = FactorAnalyzer(
            n_factors = n_factors,
            rotation  = None if n_factors == 1 else "oblimin",
            method    = "principal"
        )
        fa.fit(sim_matrix)
        loadings     = fa.loadings_
        eigenvals, _ = fa.get_eigenvalues()
        return loadings, eigenvals
    except Exception as e:
        logger.warning(f"FA failed (n={n}, k={n_factors}): {e}")
        return None, None

def residual_stats(sim_matrix: np.ndarray, loadings: np.ndarray) -> Dict:
    if loadings is None:
        return {"rmsr": 1.0, "residual_mean": 0, "residual_std": 0,
                "residual_max_abs": 0, "n_large_residuals": 0, "fit_acceptable": False}
    n_factors  = loadings.shape[1] if loadings.ndim > 1 else 1
    reproduced = (
        loadings @ loadings.T if n_factors > 1
        else np.outer(loadings.flatten(), loadings.flatten())
    )
    residuals = sim_matrix - reproduced
    np.fill_diagonal(residuals, 0)
    off = residuals[np.triu_indices_from(residuals, k=1)]
    rmsr = float(np.sqrt(np.mean(off**2)))
    return {
        "rmsr":               round(rmsr,                       4),
        "residual_mean":      round(float(np.mean(off)),        4),
        "residual_std":       round(float(np.std(off)),         4),
        "residual_max_abs":   round(float(np.max(np.abs(off))), 4),
        "n_large_residuals":  int(np.sum(np.abs(off) > 0.10)),
        "fit_acceptable":     rmsr <= RMSR_THRESHOLD
    }


# ==============================================================================
# ITEM-LEVEL RULE EVALUATION
# ==============================================================================
def evaluate_item_rules(
    atomic_single_load: float,
    communality:        float,
    max_sec_load:       float,
    within_load:        float
) -> tuple:
    rules_passed = []
    pass_results = {}
    rules_def    = {
        "Rule_1_Strong_Single_Factor":     (atomic_single_load, 0.40, ">="),
        "Rule_2_Adequate_Single_Factor":   (atomic_single_load, 0.30, ">="),
        "Rule_3_Adequate_Communality":     (communality,        0.15, ">="),
        "Rule_4_Clean_Factor_Structure":   (max_sec_load,       0.35, "<"),
        "Rule_5_Cross_Difficulty_Viable":  (atomic_single_load, 0.20, ">="),
        "Rule_6_Within_Difficulty_Viable": (within_load,        0.25, ">="),
    }
    for rule, (val, thresh, op) in rules_def.items():
        if val is None:
            passed = False
        elif op == ">=":
            passed = val >= thresh
        else:
            passed = val < thresh
        pass_results[rule] = "PASS" if passed else "FAIL"
        if passed:
            rules_passed.append(rule)
    return rules_passed, pass_results


# ==============================================================================
# JOINT MULTI-SCALE ANALYSIS  (DAAL + Discriminant Validity)
# ==============================================================================
def run_joint_analysis(df: pd.DataFrame, encoding: str = "atomic") -> Dict:
    scale_counts = df.groupby("scale_name")["item_id"].count()
    valid_scales = sorted(scale_counts[scale_counts >= MIN_ITEMS_PFA].index.tolist())
    n_scales     = len(valid_scales)

    if n_scales < MIN_SCALES_JOINT:
        logger.warning(
            f"Joint analysis skipped: only {n_scales} scales with >= {MIN_ITEMS_PFA} items "
            f"(need {MIN_SCALES_JOINT})."
        )
        return {"available": False, "scales_included": valid_scales}

    df_joint     = df[df["scale_name"].isin(valid_scales)].copy().reset_index(drop=True)
    item_ids     = df_joint["item_id"].tolist()
    scale_labels = df_joint["scale_name"].tolist()

    if encoding == "macro":
        scale_vecs = {}
        for sn in valid_scales:
            sn_df = df_joint[df_joint["scale_name"] == sn]
            emb_texts, _ = get_embedding_texts(sn_df, sn)
            # Use primary model only for macro joint — consistent embedding space
            scale_vecs[sn] = encode_macro_primary(emb_texts)
        emb_matrix = np.array([scale_vecs[sn] for sn in valid_scales])
        sim_matrix = embeddings_to_sim_matrix(emb_matrix)
        item_level = False
    else:
        emb_texts_list = []
        for _, row in df_joint.iterrows():
            txt, _ = detect_and_reverse_item(
                row["item_id"], row["item_text"], row["scale_name"]
            )
            emb_texts_list.append(txt)
        sim_matrix = encode_atomic_aggregate(emb_texts_list)
        raw_embs   = encode_atomic_raw(emb_texts_list)
        item_level = True

    loadings, eigenvals = run_fa(sim_matrix, n_scales)
    if loadings is None:
        logger.warning(f"Joint {encoding} FA failed.")
        return {"available": False, "scales_included": valid_scales}

    # DAAL factor labelling
    factor_labels = {}
    scale_daal    = {}
    if item_level:
        for sn in valid_scales:
            mask = [i for i, s in enumerate(scale_labels) if s == sn]
            scale_daal[sn] = {
                f: float(np.mean(np.abs(loadings[mask, f])))
                for f in range(n_scales)
            }
        for f in range(n_scales):
            daal_scores      = {sn: scale_daal[sn][f] for sn in valid_scales}
            factor_labels[f] = max(daal_scores, key=daal_scores.get)
    else:
        for si, sn in enumerate(valid_scales):
            scale_daal[sn] = {f: float(np.abs(loadings[si, f])) for f in range(n_scales)}
        for f in range(n_scales):
            daal_scores      = {sn: scale_daal[sn][f] for sn in valid_scales}
            factor_labels[f] = max(daal_scores, key=daal_scores.get)

    from collections import Counter as _Counter
    label_counts = _Counter(factor_labels.values())
    for f in list(factor_labels):
        if label_counts[factor_labels[f]] > 1:
            factor_labels[f] = f"Unassigned_F{f+1}"

    scale_to_factor = {v: k for k, v in factor_labels.items()
                       if not v.startswith("Unassigned")}

    daal_identity = {}
    for sn in valid_scales:
        if sn not in scale_to_factor:
            daal_identity[sn] = False
            continue
        expected_f = scale_to_factor[sn]
        if item_level:
            mask             = [i for i, s in enumerate(scale_labels) if s == sn]
            mean_on_expected = float(np.mean(np.abs(loadings[mask, expected_f])))
            all_means        = [float(np.mean(np.abs(loadings[mask, f]))) for f in range(n_scales)]
            daal_identity[sn] = (mean_on_expected == max(all_means))
        else:
            si = valid_scales.index(sn)
            daal_identity[sn] = (np.argmax(np.abs(loadings[si, :])) == expected_f)

    item_cross_loadings: Dict[str, Dict] = {}
    if item_level:
        for i, iid in enumerate(item_ids):
            item_cross_loadings[iid] = {
                factor_labels.get(f, f"F{f+1}"): round(float(loadings[i, f]), 4)
                for f in range(n_scales)
            }

    fit = residual_stats(sim_matrix, loadings)

    result = {
        "available":           True,
        "scales_included":     valid_scales,
        "factor_labels":       factor_labels,
        "scale_to_factor":     scale_to_factor,
        "scale_daal":          scale_daal,
        "daal_identity":       daal_identity,
        "item_cross_loadings": item_cross_loadings,
        "loadings":            loadings,
        "eigenvalues":         eigenvals,
        "rmsr":                fit["rmsr"],
        "fit_acceptable":      fit["fit_acceptable"],
        "item_level":          item_level,
        "item_ids":            item_ids,
        "scale_labels":        scale_labels,
    }
    if item_level:
        result["embeddings"] = raw_embs
    return result


def compute_discriminant_metrics(
    item_id:      str,
    target_scale: str,
    joint_atomic: Dict,
) -> Dict:
    empty = {
        "discriminant_available":           False,
        "discriminant_target_loading":      None,
        "discriminant_max_cross_loading":   None,
        "discriminant_target_to_max_ratio": None,
        "discriminant_closest_scale":       None,
        "discriminant_ceiling_exceeded":    None,
        "discriminant_pass":                None,
        "discriminant_flag_review":         False,
    }
    if not joint_atomic.get("available") or not joint_atomic.get("item_level"):
        return empty
    cross = joint_atomic["item_cross_loadings"].get(item_id)
    if cross is None:
        return empty
    target_loading = cross.get(target_scale)
    if target_loading is None:
        return empty

    other = {k: abs(v) for k, v in cross.items()
             if k != target_scale and not k.startswith("Unassigned")}
    if not other:
        return {**empty,
                "discriminant_available":           True,
                "discriminant_target_loading":      round(target_loading, 4),
                "discriminant_max_cross_loading":   0.0,
                "discriminant_target_to_max_ratio": None,
                "discriminant_closest_scale":       None,
                "discriminant_ceiling_exceeded":    False,
                "discriminant_pass":                True,
                "discriminant_flag_review":         False}

    max_cross_scale   = max(other, key=other.get)
    max_cross_loading = other[max_cross_scale]
    abs_target        = abs(target_loading)
    ratio             = round(abs_target / max_cross_loading, 3) if max_cross_loading > 0 else None
    ceiling_exceeded  = (max_cross_loading > DISC_CEILING) if DISC_CEIL_ENABLED else False
    disc_pass         = (ratio is not None and ratio >= DISC_RATIO_MIN) and (not ceiling_exceeded)

    return {
        "discriminant_available":           True,
        "discriminant_target_loading":      round(target_loading, 4),
        "discriminant_max_cross_loading":   round(max_cross_loading, 4),
        "discriminant_target_to_max_ratio": ratio,
        "discriminant_closest_scale":       max_cross_scale,
        "discriminant_ceiling_exceeded":    ceiling_exceeded,
        "discriminant_pass":                disc_pass,
        "discriminant_flag_review":         not disc_pass,
    }


# ==============================================================================
# LOADING MATRIX BUILDER
# ==============================================================================
def build_loading_matrix(df_source: pd.DataFrame, joint_atomic: Dict) -> pd.DataFrame:
    if not joint_atomic.get("available") or not joint_atomic.get("item_level"):
        return pd.DataFrame()

    factor_labels   = joint_atomic["factor_labels"]
    n_factors       = len(factor_labels)
    loadings_matrix = joint_atomic["loadings"]
    joint_embs      = joint_atomic["embeddings"]
    id_to_idx       = {iid: i for i, iid in enumerate(joint_atomic["item_ids"])}
    existing_ids    = set(joint_atomic["item_ids"])

    df_excl        = df_source[~df_source["item_id"].isin(existing_ids)].copy()
    excl_projected = {}
    if len(df_excl) > 0:
        excl_embs  = encode_atomic_raw(df_excl["item_text"].tolist())
        excl_norm  = excl_embs  / (np.linalg.norm(excl_embs,  axis=1, keepdims=True) + 1e-9)
        joint_norm = joint_embs / (np.linalg.norm(joint_embs, axis=1, keepdims=True) + 1e-9)
        projected  = (excl_norm @ joint_norm.T) @ loadings_matrix
        for i, iid in enumerate(df_excl["item_id"].tolist()):
            excl_projected[iid] = projected[i]

    rows = []
    for _, row in df_source.iterrows():
        iid  = row["item_id"]
        base = {
            "item_id":                     iid,
            "scale_name":                  row.get("scale_name", ""),
            "item_text":                   row.get("item_text", ""),
            "difficulty_target":           row.get("difficulty_target", ""),
            "cv_pass":                     row.get("cv_pass", False),
            "in_pfa_solution":             iid in existing_ids,
            "item_reversed_for_embedding": _reversed_flags.get(iid, False),
        }
        vec = (loadings_matrix[id_to_idx[iid]] if iid in id_to_idx
               else excl_projected.get(iid, np.zeros(n_factors)))
        for f in range(n_factors):
            label = factor_labels.get(f, f"F{f+1}")
            base[f"loading_{label}"] = round(float(vec[f]), 4)
        rows.append(base)

    return pd.DataFrame(rows)


# ==============================================================================
# MAIN PIPELINE
# ==============================================================================
print(f"\n{'='*60}")
print(f"PHASE 5: Pseudo-Factor Analysis")
print(f"Transformers     : {TRANSFORMER_MODELS}")
print(f"Min rules to pass: {MIN_RULES}")
print(f"Items entering PFA: {len(df_p6)}")
print(f"Discriminant ratio: >= {DISC_RATIO_MIN}")
print(f"Discriminant ceil : {'<= ' + str(DISC_CEILING) if DISC_CEIL_ENABLED else 'disabled'}")
print(f"RMSR threshold    : <= {RMSR_THRESHOLD}")
print(f"{'='*60}\n")

# ── Step 0: Reversal detection ────────────────────────────────────────────────
print("Running negative item detection pass...")
n_reversed = 0
for _, row in tqdm(df_p6.iterrows(), total=len(df_p6), desc="Reversal check"):
    _, was_rev = detect_and_reverse_item(
        row["item_id"], row["item_text"], row.get("scale_name", "")
    )
    if was_rev:
        n_reversed += 1
print(f"  ✓ {n_reversed} items flagged as low-pole and reversed for embedding")

# ── Step 1: Joint multi-scale analyses ───────────────────────────────────────
print("\nRunning joint multi-scale analyses...")
joint_atomic = run_joint_analysis(df_p6, encoding="atomic")
joint_macro  = run_joint_analysis(df_p6, encoding="macro")

if joint_atomic.get("available"):
    print(f"  ✓ Atomic: {len(joint_atomic['scales_included'])} scales, "
          f"RMSR={joint_atomic['rmsr']:.4f}")
    n_confirmed = sum(joint_atomic["daal_identity"].values())
    print(f"    DAAL confirmed: {n_confirmed}/{len(joint_atomic['scales_included'])} scales")
else:
    print("  ⚠ Atomic joint analysis not available")

if joint_macro.get("available"):
    print(f"  ✓ Macro : {len(joint_macro['scales_included'])} scales, "
          f"RMSR={joint_macro['rmsr']:.4f}")
else:
    print("  ⚠ Macro joint analysis not available")

# ── Step 1b: Loading matrices ─────────────────────────────────────────────────
print("\nBuilding loading matrices...")
df_loading_matrix_all     = build_loading_matrix(df_cv_all, joint_atomic)
df_loading_matrix_cv_pass = build_loading_matrix(df_p6,     joint_atomic)
if not df_loading_matrix_all.empty:
    print(f"  ✓ Loading_Matrix_All     : {len(df_loading_matrix_all)} items")
    print(f"  ✓ Loading_Matrix_CV_Pass : {len(df_loading_matrix_cv_pass)} items")

# ── Step 2: Per-scale analyses ────────────────────────────────────────────────
print("\nRunning per-scale analyses...")

pfa_rows:        List[Dict] = []
fit_stat_rows:   List[Dict] = []
eigenvalue_rows: List[Dict] = []
congruence_rows: List[Dict] = []

for scale_name, scale_grp in tqdm(df_p6.groupby("scale_name"), desc="Scales"):
    items     = scale_grp.reset_index(drop=True)
    ids_all   = items["item_id"].tolist()
    scale_def = scale_params_lookup.get(scale_name, {}).get("construct_definition", "")

    if len(ids_all) < MIN_ITEMS_PFA:
        logger.warning(f"{scale_name}: only {len(ids_all)} items — skipping.")
        continue

    emb_texts, rev_flags = get_embedding_texts(items, scale_name)

    # Atomic aggregate sim matrix + FA
    atomic_sim = encode_atomic_aggregate(emb_texts)
    a_ld1, a_eig = run_fa(atomic_sim, 1)
    a_single     = a_ld1.flatten() if a_ld1 is not None else np.zeros(len(emb_texts))
    a_fit        = residual_stats(atomic_sim, a_ld1)

    # Two-factor for secondary loading check
    a_ld2 = None
    if len(emb_texts) >= 6:
        a_ld2, _ = run_fa(atomic_sim, 2)

    # Macro aggregate item-to-scale similarities
    macro_item_sims = encode_macro_aggregate(emb_texts)

    # Tucker's congruence: atomic loadings vs macro similarities
    if a_ld1 is not None and len(macro_item_sims) == len(a_single):
        cc_values = tucker_congruence(
            a_single.reshape(-1, 1),
            macro_item_sims.reshape(-1, 1)
        )
        cc_val   = cc_values[0] if cc_values else None
        cc_label = (
            "Excellent (>= 0.95)" if cc_val is not None and cc_val >= 0.95 else
            "Fair (>= 0.85)"      if cc_val is not None and cc_val >= 0.85 else
            "Poor (< 0.85)"       if cc_val is not None else "N/A"
        )
        congruence_rows.append({
            "scale_name":                scale_name,
            "n_items":                   len(ids_all),
            "tucker_cc_atomic_vs_macro": round(cc_val, 4) if cc_val is not None else None,
            "tucker_interpretation":     cc_label,
            "note": ("Congruence between atomic aggregate single-factor loadings and macro "
                     "item-to-scale similarities. >= 0.95 excellent; >= 0.85 fair. "
                     "Lorenzo-Seva & ten Berge (2006).")
        })

    # Within-difficulty loadings
    diff_loadings: Dict[str, Dict] = {}
    for diff in ["high", "moderate", "low"]:
        diff_items = items[items["difficulty_target"] == diff].reset_index(drop=True)
        if len(diff_items) >= MIN_ITEMS_PFA:
            d_emb_texts, _ = get_embedding_texts(diff_items, scale_name)
            d_sim           = encode_atomic_aggregate(d_emb_texts)
            d_ld, _         = run_fa(d_sim, 1)
            if d_ld is not None:
                diff_loadings[diff] = dict(zip(diff_items["item_id"].tolist(), d_ld.flatten()))

    # Eigenvalues
    if a_eig is not None:
        total_eig = float(np.sum(a_eig))
        for ei, ev in enumerate(a_eig[:10]):
            eigenvalue_rows.append({
                "scale_name":          scale_name,
                "encoding":            "atomic_aggregate",
                "eigenvalue_number":   ei + 1,
                "eigenvalue":          round(float(ev), 4),
                "above_1":             "Yes" if ev > 1.0 else "No",
                "cumulative_variance": round(float(np.sum(a_eig[:ei+1])) / total_eig, 4)
                                       if total_eig > 0 else 0
            })

    # Fit stats
    fit_stat_rows.append({
        "scale_name":              scale_name,
        "construct_definition":    scale_def,
        "encoding":                "atomic_aggregate",
        "n_items":                 len(ids_all),
        "n_transformers":          len(TRANSFORMER_MODELS),
        "transformer_models":      ", ".join(TRANSFORMER_MODELS),
        "rmsr":                    a_fit["rmsr"],
        "residual_mean":           a_fit["residual_mean"],
        "residual_std":            a_fit["residual_std"],
        "residual_max_abs":        a_fit["residual_max_abs"],
        "n_large_residuals":       a_fit["n_large_residuals"],
        "fit_acceptable":          a_fit["fit_acceptable"],
        "NOTE_cfi_tli_rmsea":      "NOT COMPUTED — unreliable without sample size (Guenole et al. 2025)",
        "daal_identity_confirmed": joint_atomic.get("daal_identity", {}).get(scale_name, None),
        "n_items_reversed":        sum(rev_flags),
    })

    # Per-item evaluation
    scale_high = scale_std = scale_fail = 0

    for i, row in items.iterrows():
        iid  = row["item_id"]
        diff = row.get("difficulty_target", "high")

        a_single_load = round(float(a_single[i]), 4) if i < len(a_single) else 0.0
        communality   = round(float(a_single_load**2), 4)
        within_load   = round(float(diff_loadings.get(diff, {}).get(iid, 0.0)), 4)

        if a_ld2 is not None and i < len(a_ld2):
            vec       = a_ld2[i]
            a_primary = round(float(np.max(np.abs(vec))), 4)
            a_max_sec = round(float(np.min(np.abs(vec))), 4)
        else:
            a_primary = a_single_load
            a_max_sec = 0.0

        macro_load     = round(float(macro_item_sims[i]), 4)
        disc           = compute_discriminant_metrics(iid, scale_name, joint_atomic)
        daal_confirmed = joint_atomic.get("daal_identity", {}).get(scale_name, None)

        rules_passed, pass_results = evaluate_item_rules(
            a_single_load, communality, a_max_sec, within_load
        )

        n_passed        = len(rules_passed)
        ready_for_trial = n_passed >= MIN_RULES
        high_pass       = (
            "Rule_1_Strong_Single_Factor"   in rules_passed and
            "Rule_3_Adequate_Communality"   in rules_passed and
            "Rule_4_Clean_Factor_Structure" in rules_passed
        )
        pass_level = "HIGH_PASS" if high_pass else ("STANDARD_PASS" if ready_for_trial else "FAIL")

        if pass_level == "HIGH_PASS":       scale_high += 1
        elif pass_level == "STANDARD_PASS": scale_std  += 1
        else:                               scale_fail += 1

        pfa_row = {
            "item_id":                        iid,
            "scale_name":                     scale_name,
            "item_text":                      row["item_text"],
            "item_reversed_for_embedding":    rev_flags[i],
            "difficulty_target":              diff,
            "occurrence_likelihood":          row.get("occurrence_likelihood",       ""),
            "source_example":                 row.get("source_example",              ""),
            "flesch_kincaid_grade":           row.get("flesch_kincaid_grade",        0),
            "was_simplified":                 row.get("was_simplified",              False),
            "simplification_attempts":        row.get("simplification_attempts",     0),
            "bias_flags":                     row.get("bias_flags",                  ""),
            "bias_flag_count":                row.get("bias_flag_count",             0),
            "cv_mean_target_rating":          row.get("cv_mean_target_rating",       0),
            "cv_std_target_rating":           row.get("cv_std_target_rating",        0),
            "cv_scale_mapping_agreement":     row.get("cv_scale_mapping_agreement",  0),
            "cv_maps_to_target_count":        row.get("cv_maps_to_target_count",     0),
            "cv_maps_to_target":              row.get("cv_maps_to_target",           False),
            "cv_majority_mapped_scale":       row.get("cv_majority_mapped_scale",    ""),
            "cv_majority_agreement":          row.get("cv_majority_agreement",       0),
            "cv_majority_is_target":          row.get("cv_majority_is_target",       False),
            "cv_sme_ratings":                 row.get("cv_sme_ratings",              ""),
            "cv_sme_mappings":                row.get("cv_sme_mappings",             ""),
            "cv_pass":                        row.get("cv_pass",                     False),
            "pfa_atomic_single_loading":      a_single_load,
            "pfa_atomic_communality":         communality,
            "pfa_atomic_primary_loading":     a_primary,
            "pfa_atomic_max_sec_loading":     a_max_sec,
            "pfa_atomic_within_diff_loading": within_load,
            "pfa_macro_item_to_scale_sim":    macro_load,
            "daal_identity_confirmed":        daal_confirmed,
            **disc,
            "pfa_scale_rmsr":                 a_fit["rmsr"],
            "pfa_fit_acceptable":             a_fit["fit_acceptable"],
            "pfa_rules_passed_count":         n_passed,
            "pfa_rules_failed_count":         6 - n_passed,
            "pfa_ready_for_trial":            ready_for_trial,
            "pfa_pass_level":                 pass_level,
        }
        for rule, result in pass_results.items():
            pfa_row[rule] = result

        pfa_rows.append(pfa_row)

    print(
        f"  {scale_name:<35}  n={len(items):>3}  rev={sum(rev_flags):>2}  "
        f"HIGH={scale_high}  STD={scale_std}  FAIL={scale_fail}  "
        f"RMSR={a_fit['rmsr']:.3f}  "
        f"DAAL={'✓' if joint_atomic.get('daal_identity',{}).get(scale_name) else ('✗' if joint_atomic.get('available') else '?')}"
    )

df_pfa = pd.DataFrame(pfa_rows)


# ==============================================================================
# DISCRIMINANT MATRIX — all items, all factors, factor-index order (REVERTED)
# Includes Unassigned factors. No filtering.
# ==============================================================================
disc_matrix_rows: List[Dict] = []
if joint_atomic.get("available") and joint_atomic.get("item_level"):
    factor_labels      = joint_atomic["factor_labels"]
    ordered_factor_cols = [
        factor_labels.get(f, f"F{f+1}")
        for f in range(len(factor_labels))
    ]
    for iid, cross in joint_atomic["item_cross_loadings"].items():
        row_data = df_pfa[df_pfa["item_id"] == iid]
        if row_data.empty:
            continue
        base_row = {
            "item_id":    iid,
            "scale_name": row_data.iloc[0]["scale_name"],
            "item_text":  row_data.iloc[0]["item_text"]
        }
        for col in ordered_factor_cols:
            base_row[f"loading_on_{col}"] = cross.get(col, None)
        disc_matrix_rows.append(base_row)
df_disc_matrix = pd.DataFrame(disc_matrix_rows)


# ==============================================================================
# ITEM-SCALE HEATMAP — cosine similarity, scales.json order, all items
# Uses primary model only for both item and scale embeddings — consistent dims.
# Every item vs every scale regardless of DAAL or cv_pass status.
# ==============================================================================
print("\nBuilding Item-Scale Heatmap...")

# Scale order from scales.json
heatmap_scale_order = [s["scale_name"] for s in TARGET_SCALES]

# Macro embedding per scale using primary model only
scale_primary_vecs = {}
for sn in heatmap_scale_order:
    scale_items = df_p6[df_p6["scale_name"] == sn]["item_text"].tolist()
    if not scale_items:
        scale_items = df_cv_all[df_cv_all["scale_name"] == sn]["item_text"].tolist()
    if scale_items:
        preprocessed = [preprocess_for_embedding(t) for t in scale_items]
        concatenated  = " ".join(preprocessed)
        vec = PRIMARY_MODEL.encode([concatenated], show_progress_bar=False)[0]
        scale_primary_vecs[sn] = vec / (np.linalg.norm(vec) + 1e-9)
    else:
        scale_primary_vecs[sn] = None

# Item embeddings using primary model — all PFA items
all_item_texts = df_pfa["item_text"].tolist()
all_item_embs  = encode_atomic_raw(all_item_texts)
all_item_norms = all_item_embs / (
    np.linalg.norm(all_item_embs, axis=1, keepdims=True) + 1e-9
)

# Build heatmap rows
heatmap_rows = []
for i, (_, row) in enumerate(df_pfa.iterrows()):
    hmap_row = {
        "item_id":        row["item_id"],
        "scale_name":     row["scale_name"],
        "item_text":      row["item_text"],
        "pfa_pass_level": row["pfa_pass_level"],
    }
    item_norm = all_item_norms[i]
    for sn in heatmap_scale_order:
        if scale_primary_vecs.get(sn) is not None:
            hmap_row[f"sim_{sn}"] = round(float(item_norm @ scale_primary_vecs[sn]), 4)
        else:
            hmap_row[f"sim_{sn}"] = None
    heatmap_rows.append(hmap_row)

df_heatmap = pd.DataFrame(heatmap_rows)

# Sort rows in scales.json order
scale_order_map        = {sn: i for i, sn in enumerate(heatmap_scale_order)}
df_heatmap["_sort"]    = df_heatmap["scale_name"].map(scale_order_map).fillna(999)
df_heatmap             = df_heatmap.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

print(f"  ✓ Item-Scale Heatmap: {len(df_heatmap)} items × {len(heatmap_scale_order)} scales")


# ==============================================================================
# SCALE SUMMARY
# ==============================================================================
agg_dict = {
    "total_items":              ("item_id",                     "count"),
    "high_pass":                ("pfa_pass_level",  lambda x: (x == "HIGH_PASS").sum()),
    "standard_pass":            ("pfa_pass_level",  lambda x: (x == "STANDARD_PASS").sum()),
    "fail":                     ("pfa_pass_level",  lambda x: (x == "FAIL").sum()),
    "n_reversed_for_embedding": ("item_reversed_for_embedding", "sum"),
    "mean_atomic_loading":      ("pfa_atomic_single_loading",   "mean"),
    "mean_communality":         ("pfa_atomic_communality",       "mean"),
    "mean_macro_sim":           ("pfa_macro_item_to_scale_sim",  "mean"),
    "mean_cv_rating":           ("cv_mean_target_rating",        "mean"),
    "mean_cv_agreement":        ("cv_scale_mapping_agreement",   "mean"),
    "mean_scale_rmsr":          ("pfa_scale_rmsr",               "mean"),
}
if "discriminant_pass" in df_pfa.columns and df_pfa["discriminant_pass"].notna().any():
    agg_dict["disc_pass_count"] = ("discriminant_pass",        lambda x: x.sum() if x.notna().any() else 0)
    agg_dict["disc_flag_count"] = ("discriminant_flag_review", lambda x: x.sum() if x.notna().any() else 0)

pfa_scale_summary = df_pfa.groupby("scale_name").agg(**agg_dict).reset_index()

for col in ["mean_atomic_loading", "mean_communality", "mean_macro_sim",
            "mean_cv_rating", "mean_cv_agreement", "mean_scale_rmsr"]:
    if col in pfa_scale_summary.columns:
        pfa_scale_summary[col] = pfa_scale_summary[col].round(3)

pfa_scale_summary["pass_rate_pct"] = round(
    100 * (pfa_scale_summary["high_pass"] + pfa_scale_summary["standard_pass"])
    / pfa_scale_summary["total_items"], 1
)

if joint_atomic.get("available"):
    daal_df = pd.DataFrame([
        {"scale_name": k, "daal_identity_confirmed": v}
        for k, v in joint_atomic["daal_identity"].items()
    ])
    pfa_scale_summary = pfa_scale_summary.merge(daal_df, on="scale_name", how="left")

if congruence_rows:
    cc_df = pd.DataFrame(congruence_rows)[
        ["scale_name", "tucker_cc_atomic_vs_macro", "tucker_interpretation"]
    ]
    pfa_scale_summary = pfa_scale_summary.merge(cc_df, on="scale_name", how="left")


# ==============================================================================
# PASS RULES DOCUMENTATION
# ==============================================================================
rules_def_rows = [
    {"rule_id": k, "description": v["description"],
     "threshold": v["threshold"], "operator": v["operator"]}
    for k, v in PASS_RULES_CFG.items()
]
rules_def_rows += [
    {"rule_id": "min_rules_to_pass",
     "description": "Minimum rules passed to be ready for trial",
     "threshold": MIN_RULES, "operator": ">="},
    {"rule_id": "discriminant_ratio_min",
     "description": "Min target/max-cross loading ratio for discriminant pass",
     "threshold": DISC_RATIO_MIN, "operator": ">="},
    {"rule_id": "discriminant_ceiling",
     "description": "Max absolute cross-loading before ceiling flag",
     "threshold": DISC_CEILING, "operator": "<"},
    {"rule_id": "rmsr_max",
     "description": "Max RMSR for fit_acceptable (Guenole et al. 2025)",
     "threshold": RMSR_THRESHOLD, "operator": "<="},
    {"rule_id": "tucker_excellent",
     "description": "Tucker CC >= 0.95 = excellent encoding invariance",
     "threshold": 0.95, "operator": ">="},
    {"rule_id": "tucker_fair",
     "description": "Tucker CC >= 0.85 = fair encoding invariance (Lorenzo-Seva & ten Berge 2006)",
     "threshold": 0.85, "operator": ">="},
    {"rule_id": "NOTE_cfi_tli_rmsea",
     "description": "CFI/TLI/RMSEA NOT USED — require sample size (Guenole et al. 2025)",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_reversal",
     "description": "Low-pole items reversed for embedding only. item_text always original.",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_transformers",
     "description": f"Similarity matrices aggregated across: {', '.join(TRANSFORMER_MODELS)}",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_heatmap",
     "description": "Item_Scale_Heatmap uses primary model only for consistent dimensionality",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_projected",
     "description": "Loading_Matrix_All: in_pfa_solution=False items have projected loadings",
     "threshold": "N/A", "operator": "N/A"},
]

# ==============================================================================
# SUBSET SHEETS
# ==============================================================================
keep_cols = [
    "item_id", "scale_name", "item_text", "item_reversed_for_embedding",
    "difficulty_target", "flesch_kincaid_grade", "cv_mean_target_rating",
    "pfa_atomic_single_loading", "pfa_macro_item_to_scale_sim",
    "pfa_atomic_max_sec_loading", "daal_identity_confirmed",
    "discriminant_pass", "discriminant_flag_review"
]
keep_cols    = [c for c in keep_cols if c in df_pfa.columns]
df_high_pass = df_pfa[df_pfa["pfa_pass_level"] == "HIGH_PASS"][keep_cols].copy()
df_all_pass  = df_pfa[df_pfa["pfa_ready_for_trial"] == True][keep_cols].copy()


# ==============================================================================
# PRINT SUMMARY
# ==============================================================================
n_high = (df_pfa["pfa_pass_level"] == "HIGH_PASS").sum()
n_std  = (df_pfa["pfa_pass_level"] == "STANDARD_PASS").sum()
n_fail = (df_pfa["pfa_pass_level"] == "FAIL").sum()

print(f"\n{'='*60}")
print("PFA Scale Summary")
print(f"{'='*60}")
summary_cols = ["scale_name", "total_items", "n_reversed_for_embedding",
                "high_pass", "standard_pass", "fail", "pass_rate_pct",
                "mean_atomic_loading", "mean_scale_rmsr"]
if "daal_identity_confirmed" in pfa_scale_summary.columns:
    summary_cols.append("daal_identity_confirmed")
if "tucker_cc_atomic_vs_macro" in pfa_scale_summary.columns:
    summary_cols += ["tucker_cc_atomic_vs_macro", "tucker_interpretation"]
print(pfa_scale_summary[summary_cols].to_string(index=False))


# ==============================================================================
# WRITE EXCEL
# ==============================================================================
df_pfa["human_review_pass"] = True
df_pfa["human_comments"]    = ""

with pd.ExcelWriter(P6_OUT, engine="openpyxl") as writer:
    df_pfa.to_excel(                           writer, sheet_name="PFA_Item_Results",       index=False)
    pfa_scale_summary.to_excel(                writer, sheet_name="PFA_Scale_Summary",      index=False)
    if not df_disc_matrix.empty:
        df_disc_matrix.to_excel(               writer, sheet_name="Discriminant_Matrix",    index=False)
    if not df_heatmap.empty:
        df_heatmap.to_excel(                   writer, sheet_name="Item_Scale_Heatmap",     index=False)
    if not df_loading_matrix_all.empty:
        df_loading_matrix_all.to_excel(        writer, sheet_name="Loading_Matrix_All",     index=False)
        df_loading_matrix_cv_pass.to_excel(    writer, sheet_name="Loading_Matrix_CV_Pass", index=False)
    pd.DataFrame(fit_stat_rows).to_excel(      writer, sheet_name="Fit_Statistics",         index=False)
    pd.DataFrame(eigenvalue_rows).to_excel(    writer, sheet_name="Eigenvalues",            index=False)
    if congruence_rows:
        pd.DataFrame(congruence_rows).to_excel(writer, sheet_name="Tucker_Congruence",      index=False)
    df_high_pass.to_excel(                     writer, sheet_name="High_Pass_Loadings",     index=False)
    df_all_pass.to_excel(                      writer, sheet_name="All_Pass_Loadings",      index=False)
    pd.DataFrame(rules_def_rows).to_excel(     writer, sheet_name="Pass_Rules_Definition",  index=False)


# ==============================================================================
# FINAL PRINT
# ==============================================================================
print(f"\n✅ Phase 5 PFA complete.")
print(f"   HIGH_PASS    : {n_high}")
print(f"   STANDARD_PASS: {n_std}")
print(f"   FAIL         : {n_fail}")
print(f"   Total in PFA : {len(df_pfa)} "
      f"(from {len(df_cv_all)} Phase 5 items, {n_excluded} excluded)")
print(f"   Items reversed for embedding : {n_reversed}")
print(f"   Transformers aggregated      : {len(TRANSFORMER_MODELS)} ({', '.join(TRANSFORMER_MODELS)})")
if joint_atomic.get("available"):
    n_daal = sum(joint_atomic["daal_identity"].values())
    n_tot  = len(joint_atomic["daal_identity"])
    print(f"   DAAL identity: {n_daal}/{n_tot} scales confirmed")
if "discriminant_flag_review" in df_pfa.columns:
    print(f"   Discriminant flags: {int(df_pfa['discriminant_flag_review'].sum())} items flagged")
if not df_heatmap.empty:
    print(f"   Item_Scale_Heatmap: {len(df_heatmap)} items × {len(heatmap_scale_order)} scales")
if congruence_rows:
    excellent = sum(1 for r in congruence_rows if (r.get("tucker_cc_atomic_vs_macro") or 0) >= 0.95)
    fair      = sum(1 for r in congruence_rows if 0.85 <= (r.get("tucker_cc_atomic_vs_macro") or 0) < 0.95)
    poor      = sum(1 for r in congruence_rows if (r.get("tucker_cc_atomic_vs_macro") or 0) < 0.85)
    print(f"   Tucker's congruence: {excellent} excellent / {fair} fair / {poor} poor")
if not df_loading_matrix_all.empty:
    n_proj = (~df_loading_matrix_all["in_pfa_solution"]).sum()
    print(f"   Loading_Matrix_All: {len(df_loading_matrix_all)} items ({n_proj} projected)")
print(f"📄 Output: {P6_OUT}")


# ==============================================================================
# PHASE 5 (continued) — Item Pool Assembly & Human Review Package
# ==============================================================================

# ==============================================================================
# PHASE 5 (continued) — Item Pool Assembly & Human Review Package
# ==============================================================================

# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

P7   = P6  # Item pool assembly settings are now part of Phase 5
P4   = CFG["phase_03_readability_bias"]
P5_CV = CFG["phase_04_content_validity"]

# Item pool assembly reads from the PFA results we just computed
P7_IN  = P6_OUT  # same file we just wrote
P7_OUT = P6_OUT  # we write additional sheets into the same file

assert P7_IN.exists(), f"PFA output not found: {P7_IN}. PFA must complete first."

df_p7 = pd.read_excel(P7_IN, sheet_name="PFA_Item_Results").copy()
print(f"✓ PFA items loaded: {len(df_p7)}")

# ── Guard: scale_name must exist ──────────────────────────────────────────────
assert "scale_name" in df_p7.columns, (
    "ERROR: 'scale_name' column not found in PFA output. "
    "Re-run Phase 5 with the current code."
)

# ── Pass logic ────────────────────────────────────────────────────────────────
# Use human_review_pass as the definitive gate.
# != False means NaN values pass through — only an explicit False excludes an item.
# Falls back to system pfa + cv flags if the column is absent.
if "human_review_pass" in df_p7.columns:
    df_p7["overall_pass"] = (df_p7["human_review_pass"] != False)
    print("✓ PFA human_review_pass column found — using as primary gate")
else:
    df_p7["pfa_passed"]   = df_p7["pfa_ready_for_trial"].fillna(False).astype(bool)
    df_p7["cv_passed"]    = df_p7["cv_pass"].fillna(False).astype(bool)
    df_p7["overall_pass"] = df_p7["pfa_passed"] & df_p7["cv_passed"]
    print("⚠ human_review_pass not found — falling back to system pfa_passed & cv_passed")

df_pass   = df_p7[df_p7["overall_pass"]].copy().reset_index(drop=True)
df_reject = df_p7[~df_p7["overall_pass"]].copy().reset_index(drop=True)

print(f"  Overall pass : {len(df_pass)}")
print(f"  Rejected     : {len(df_reject)}")

# ── Add human review columns ──────────────────────────────────────────────────
df_pass["human_decision"] = ""   # accept | modify | reject
df_pass["human_notes"]    = ""   # free-text rationale

# ── Build review column list ──────────────────────────────────────────────────
review_cols = P7["include_columns"]
if "scale_name" not in review_cols:
    review_cols = ["scale_name"] + review_cols

available_cols = [c for c in review_cols if c in df_pass.columns]
missing_cols   = [c for c in review_cols if c not in df_pass.columns]
if missing_cols:
    logger.warning(
        f"Phase 5 (continued): {len(missing_cols)} configured columns not found in PFA output: "
        f"{missing_cols}"
    )

if "scale_name" not in available_cols:
    available_cols = ["scale_name"] + available_cols

df_review = df_pass[available_cols].copy()
print(f"  Review columns: {len(available_cols)}")

# ── Pool statistics ───────────────────────────────────────────────────────────
pool_stats_rows: List[Dict] = []
for scale_name, grp in df_review.groupby("scale_name"):
    diff_dist      = grp["difficulty_target"].value_counts().to_dict()
    daal_col       = "daal_identity_confirmed"
    disc_col       = "discriminant_flag_review"
    rev_col        = "item_reversed_for_embedding"
    daal_confirmed = int(grp[daal_col].sum()) if daal_col in grp.columns and grp[daal_col].notna().any() else None
    disc_flags     = int(grp[disc_col].sum()) if disc_col in grp.columns and grp[disc_col].notna().any() else 0
    n_reversed     = int(grp[rev_col].sum()) if rev_col in grp.columns and grp[rev_col].notna().any() else 0

    pool_stats_rows.append({
        "scale_name":                  scale_name,
        "total_items":                 len(grp),
        "high_diff_items":             diff_dist.get("high",     0),
        "moderate_diff_items":         diff_dist.get("moderate", 0),
        "low_diff_items":              diff_dist.get("low",      0),
        "high_pass_items":             (grp["pfa_pass_level"] == "HIGH_PASS").sum(),
        "standard_pass_items":         (grp["pfa_pass_level"] == "STANDARD_PASS").sum(),
        "mean_c_value":                round(grp["cv_c_value"].mean(), 3)
                                       if "cv_c_value" in grp else 0,
        "mean_d_value":                round(grp["cv_d_value"].mean(), 3)
                                       if "cv_d_value" in grp else 0,
        "mean_cv_rating":              round(grp["cv_mean_target_rating"].mean(), 2)
                                       if "cv_mean_target_rating" in grp else 0,
        "mean_cv_agreement":           round(grp["cv_scale_mapping_agreement"].mean(), 2)
                                       if "cv_scale_mapping_agreement" in grp else 0,
        "mean_atomic_loading":         round(grp["pfa_atomic_single_loading"].mean(), 3)
                                       if "pfa_atomic_single_loading" in grp else 0,
        "mean_macro_sim":              round(grp["pfa_macro_item_to_scale_sim"].mean(), 3)
                                       if "pfa_macro_item_to_scale_sim" in grp else 0,
        "mean_fk_grade":               round(grp["flesch_kincaid_grade"].mean(), 2)
                                       if "flesch_kincaid_grade" in grp else 0,
        "items_with_bias_flags":       grp["bias_flags"].fillna("").astype(str).str.strip().str.len().gt(0).sum(),
        "items_reversed_for_embedding": n_reversed,
        "daal_identity_confirmed":     daal_confirmed,
        "discriminant_flags":          disc_flags,
    })

# ── Quality flags ─────────────────────────────────────────────────────────────
reading_target = P4.get("reading_level_target", 8)
cv_c_warn = P5_CV["pass_thresholds"].get("min_c_value", 0.88)
cv_d_warn = P5_CV["pass_thresholds"].get("min_d_value", 0.35)

quality_flag_rows: List[Dict] = []
for _, row in df_review.iterrows():
    flags = []

    if row.get("flesch_kincaid_grade", 0) > reading_target:
        flags.append(
            f"Above FK grade {reading_target} ({row['flesch_kincaid_grade']:.1f})"
        )

    if str(row.get("bias_flags", "")).strip():
        flags.append(f"Bias: {row['bias_flags']}")

    if row.get("cv_c_value", 1.0) < 0.88:
        flags.append(
            f"Low c-value ({row.get('cv_c_value', 0):.3f}, threshold: 0.88)"
        )

    if row.get("cv_d_value", 1.0) < 0.35:
        flags.append(
            f"Low d-value ({row.get('cv_d_value', 0):.3f}, threshold: 0.35)"
        )

    if row.get("cv_mean_target_rating", 10) < cv_c_warn + 1:
        flags.append(
            f"Low CV target rating ({row.get('cv_mean_target_rating', 0):.1f}/7)"
        )

    if not row.get("cv_maps_to_target", True):
        flags.append(
            f"CV mapped to '{row.get('cv_majority_mapped_scale', '?')}' not target"
        )

    if row.get("pfa_pass_level") == "STANDARD_PASS":
        flags.append("Standard pass only")

    if row.get("discriminant_flag_review") == True:
        closest   = row.get("discriminant_closest_scale", "?")
        ratio     = row.get("discriminant_target_to_max_ratio", None)
        ratio_str = f", ratio={ratio:.2f}" if ratio is not None else ""
        flags.append(f"Discriminant flag: closest to '{closest}'{ratio_str}")

    if row.get("daal_identity_confirmed") == False:
        flags.append("DAAL identity not confirmed")

    if row.get("item_reversed_for_embedding") == True:
        flags.append("Item reversed for embedding — verify semantic direction")

    if flags:
        quality_flag_rows.append({
            "item_id":        row.get("item_id"),
            "scale_name":     row.get("scale_name"),
            "item_text":      row.get("item_text"),
            "pfa_pass_level": row.get("pfa_pass_level"),
            "flags":          " | ".join(flags),
            "flag_count":     len(flags),
        })

# ── Recommendations ───────────────────────────────────────────────────────────
target_per_diff = P7.get("final_pool_target_per_difficulty", 20)
recs: List[Dict] = []

for row in pool_stats_rows:
    sn = row["scale_name"]

    if row["total_items"] < target_per_diff * 3:
        recs.append({
            "scale_name":     sn,
            "priority":       "HIGH",
            "recommendation": (
                f"Only {row['total_items']} items passed "
                f"(target: {target_per_diff * 3}). "
                f"Consider regenerating more items."
            )
        })

    for diff, key in [
        ("high",     "high_diff_items"),
        ("moderate", "moderate_diff_items"),
        ("low",      "low_diff_items")
    ]:
        if row[key] < target_per_diff:
            recs.append({
                "scale_name":     sn,
                "priority":       "MEDIUM",
                "recommendation": (
                    f"Only {row[key]} {diff}-difficulty items "
                    f"(target: {target_per_diff})."
                )
            })

    if row["mean_fk_grade"] > reading_target:
        recs.append({
            "scale_name":     sn,
            "priority":       "MEDIUM",
            "recommendation": (
                f"Mean FK grade {row['mean_fk_grade']} above target {reading_target}."
            )
        })

    if row.get("mean_c_value", 1.0) < 0.88:
        recs.append({
            "scale_name":     sn,
            "priority":       "MEDIUM",
            "recommendation": (
                f"Mean c-value {row.get('mean_c_value', 0):.3f} below 0.88 threshold."
            )
        })

    if row.get("mean_d_value", 1.0) < 0.35:
        recs.append({
            "scale_name":     sn,
            "priority":       "MEDIUM",
            "recommendation": (
                f"Mean d-value {row.get('mean_d_value', 0):.3f} below 0.35 threshold."
            )
        })

    if row["items_with_bias_flags"] > 0:
        recs.append({
            "scale_name":     sn,
            "priority":       "HIGH",
            "recommendation": (
                f"{row['items_with_bias_flags']} items have bias flags. "
                f"Prioritize review."
            )
        })

    if row.get("items_reversed_for_embedding", 0) > 0:
        recs.append({
            "scale_name":     sn,
            "priority":       "MEDIUM",
            "recommendation": (
                f"{row['items_reversed_for_embedding']} items were reversed for "
                f"embedding — verify semantic direction before trialling."
            )
        })

    if row.get("discriminant_flags", 0) > 0:
        recs.append({
            "scale_name":     sn,
            "priority":       "MEDIUM",
            "recommendation": (
                f"{row['discriminant_flags']} items flagged for discriminant validity. "
                f"Check 'discriminant_closest_scale'."
            )
        })

    if row.get("daal_identity_confirmed") == False:
        recs.append({
            "scale_name":     sn,
            "priority":       "HIGH",
            "recommendation": (
                "DAAL factor identity not confirmed. "
                "Review construct definition and Item_Scale_Heatmap."
            )
        })

priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
recs.sort(key=lambda x: priority_order.get(x.get("priority", "LOW"), 2))

# ── Rejected items ────────────────────────────────────────────────────────────
reject_cols   = [c for c in available_cols if c not in ("human_decision", "human_notes")]
df_reject_out = df_reject[[c for c in reject_cols if c in df_reject.columns]].copy()

# ── Write Excel ───────────────────────────────────────────────────────────────
# Append review sheets to existing PFA output file
from openpyxl import load_workbook
with pd.ExcelWriter(P7_OUT, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_review.to_excel(
        writer, sheet_name="Review_Pool", index=False
    )
    df_reject_out.to_excel(
        writer, sheet_name="Rejected_Items", index=False
    )
    pd.DataFrame(pool_stats_rows).to_excel(
        writer, sheet_name="Pool_Statistics", index=False
    )
    if quality_flag_rows:
        pd.DataFrame(quality_flag_rows).to_excel(
            writer, sheet_name="Quality_Flags", index=False
        )
    if recs:
        pd.DataFrame(recs).to_excel(
            writer, sheet_name="Recommendations", index=False
        )

# ── Final summary ─────────────────────────────────────────────────────────────
n_high_flags  = sum(1 for r in recs if r.get("priority") == "HIGH")
n_med_flags   = sum(1 for r in recs if r.get("priority") == "MEDIUM")
n_reversed    = int(df_review.get("item_reversed_for_embedding", pd.Series(dtype=bool)).sum()) \
                if "item_reversed_for_embedding" in df_review.columns else 0

print(f"\n{'='*60}")
print(f"✅ PIPELINE COMPLETE — Phase 5 (continued): Item Pool Assembly")
print(f"{'='*60}")
print(f"Items in review pool       : {len(df_review)}")
print(f"Items rejected             : {len(df_reject)}")
print(f"Items with QC flags        : {len(quality_flag_rows)}")
print(f"Items reversed for embedding: {n_reversed}")
print(f"Recommendations            : {len(recs)} "
      f"({n_high_flags} HIGH / {n_med_flags} MEDIUM priority)")
print(f"\n📄 Review file: {P7_OUT}")
print(f"\nReview workflow:")
print(f"  1. Open '{P7['output_file']}'")
print(f"  2. Start with 'Recommendations' and 'Quality_Flags' sheets")
print(f"  3. Work through 'Review_Pool' scale by scale")
print(f"  4. Fill in 'human_decision' (accept / modify / reject)")
print(f"  5. Add rationale in 'human_notes' — required for any reversal or flag")